# Individual modeling: baselines, tree ensembles, and neural networks


This notebook documents the individual phase of the project. It compares a naive baseline, non-deep-learning models, and neural networks using five-fold cross-validation and original-scale sMAPE. The later group phase is documented separately in `02_group_quantile_model.ipynb`.


In [1]:
from pathlib import Path
import platform
import random
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 937
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
# Repository paths
WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = REPO_ROOT / "data" / "raw"
ARTIFACT_DIR = REPO_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_individual.csv")

print(f"Training-data shape: {train.shape}")
print(f"Test-data shape:     {test.shape}")

display(train.head())
display(test.head())

Training-data shape: (100000, 212)
Test-data shape:     (50000, 203)


,fullTimeEmployees,auditRisk,boardRisk,compensationRisk,shareHolderRightsRisk,overallRisk,floatShares,sharesOutstanding,trailingEps,forwardEps,targetHighPrice,targetLowPrice,targetMeanPrice,targetMedianPrice,recommendationMean,numberOfAnalystOpinions,totalCash,totalCashPerShare,ebitda,totalDebt,totalRevenue,revenuePerShare,freeCashflow,operatingCashflow,revenueGrowth,Q0_TOTAL_ASSETS,Q0_TOTAL_LIABILITIES,Q0_TOTAL_STOCKHOLDERS_EQUITY,Q0_GROSS_PROFIT,Q0_COST_OF_REVENUES,Q0_REVENUES,Q0_OPERATING_INCOME,Q0_OPERATING_EXPENSES,Q0_EBITDA,Q0_fiscal_year_end,Q1_TOTAL_ASSETS,Q1_TOTAL_CURRENT_ASSETS,Q1_TOTAL_NONCURRENT_ASSETS,Q1_TOTAL_LIABILITIES,Q1_TOTAL_CURRENT_LIABILITIES,Q1_TOTAL_NONCURRENT_LIABILITIES,Q1_TOTAL_LIABILITIES_AND_EQUITY,Q1_TOTAL_STOCKHOLDERS_EQUITY,Q1_NET_INCOME,Q1_GROSS_PROFIT,Q1_COST_OF_REVENUES,Q1_REVENUES,Q1_OPERATING_INCOME,Q1_OPERATING_EXPENSES,Q1_EBITDA,Q1_DEPRECIATION_AND_AMORTIZATION,Q1_fiscal_year_end,Q2_TOTAL_ASSETS,Q2_TOTAL_CURRENT_ASSETS,Q2_TOTAL_NONCURRENT_ASSETS,Q2_TOTAL_LIABILITIES,Q2_TOTAL_CURRENT_LIABILITIES,Q2_TOTAL_NONCURRENT_LIABILITIES,Q2_TOTAL_LIABILITIES_AND_EQUITY,Q2_TOTAL_STOCKHOLDERS_EQUITY,Q2_NET_INCOME,Q2_GROSS_PROFIT,Q2_COST_OF_REVENUES,Q2_REVENUES,Q2_OPERATING_INCOME,Q2_OPERATING_EXPENSES,Q2_EBITDA,Q2_DEPRECIATION_AND_AMORTIZATION,Q2_fiscal_year_end,Q3_TOTAL_ASSETS,Q3_TOTAL_CURRENT_ASSETS,Q3_TOTAL_NONCURRENT_ASSETS,Q3_TOTAL_LIABILITIES,Q3_TOTAL_CURRENT_LIABILITIES,Q3_TOTAL_NONCURRENT_LIABILITIES,Q3_TOTAL_LIABILITIES_AND_EQUITY,Q3_TOTAL_STOCKHOLDERS_EQUITY,Q3_NET_INCOME,Q3_GROSS_PROFIT,Q3_COST_OF_REVENUES,Q3_REVENUES,Q3_OPERATING_INCOME,Q3_OPERATING_EXPENSES,Q3_EBITDA,Q3_DEPRECIATION_AND_AMORTIZATION,Q3_fiscal_year_end,Q4_TOTAL_ASSETS,Q4_TOTAL_CURRENT_ASSETS,Q4_TOTAL_NONCURRENT_ASSETS,Q4_TOTAL_LIABILITIES,Q4_TOTAL_CURRENT_LIABILITIES,Q4_TOTAL_NONCURRENT_LIABILITIES,Q4_TOTAL_LIABILITIES_AND_EQUITY,Q4_TOTAL_STOCKHOLDERS_EQUITY,Q4_NET_INCOME,Q4_GROSS_PROFIT,Q4_COST_OF_REVENUES,Q4_REVENUES,Q4_OPERATING_INCOME,Q4_OPERATING_EXPENSES,Q4_EBITDA,Q4_DEPRECIATION_AND_AMORTIZATION,Q4_fiscal_year_end,Q5_TOTAL_ASSETS,Q5_TOTAL_CURRENT_ASSETS,Q5_TOTAL_NONCURRENT_ASSETS,Q5_TOTAL_LIABILITIES,Q5_TOTAL_CURRENT_LIABILITIES,Q5_TOTAL_NONCURRENT_LIABILITIES,Q5_TOTAL_LIABILITIES_AND_EQUITY,Q5_TOTAL_STOCKHOLDERS_EQUITY,Q5_NET_INCOME,Q5_GROSS_PROFIT,Q5_COST_OF_REVENUES,Q5_REVENUES,Q5_OPERATING_INCOME,Q5_OPERATING_EXPENSES,Q5_EBITDA,Q5_DEPRECIATION_AND_AMORTIZATION,Q5_fiscal_year_end,Q6_TOTAL_ASSETS,Q6_TOTAL_CURRENT_ASSETS,Q6_TOTAL_NONCURRENT_ASSETS,Q6_TOTAL_LIABILITIES,Q6_TOTAL_CURRENT_LIABILITIES,Q6_TOTAL_NONCURRENT_LIABILITIES,Q6_TOTAL_LIABILITIES_AND_EQUITY,Q6_TOTAL_STOCKHOLDERS_EQUITY,Q6_NET_INCOME,Q6_GROSS_PROFIT,Q6_COST_OF_REVENUES,Q6_REVENUES,Q6_OPERATING_INCOME,Q6_OPERATING_EXPENSES,Q6_EBITDA,Q6_DEPRECIATION_AND_AMORTIZATION,Q6_fiscal_year_end,Q7_TOTAL_ASSETS,Q7_TOTAL_CURRENT_ASSETS,Q7_TOTAL_NONCURRENT_ASSETS,Q7_TOTAL_LIABILITIES,Q7_TOTAL_CURRENT_LIABILITIES,Q7_TOTAL_NONCURRENT_LIABILITIES,Q7_TOTAL_LIABILITIES_AND_EQUITY,Q7_TOTAL_STOCKHOLDERS_EQUITY,Q7_NET_INCOME,Q7_GROSS_PROFIT,Q7_COST_OF_REVENUES,Q7_REVENUES,Q7_OPERATING_INCOME,Q7_OPERATING_EXPENSES,Q7_EBITDA,Q7_DEPRECIATION_AND_AMORTIZATION,Q7_fiscal_year_end,Q8_TOTAL_ASSETS,Q8_TOTAL_CURRENT_ASSETS,Q8_TOTAL_NONCURRENT_ASSETS,Q8_TOTAL_LIABILITIES,Q8_TOTAL_CURRENT_LIABILITIES,Q8_TOTAL_NONCURRENT_LIABILITIES,Q8_TOTAL_LIABILITIES_AND_EQUITY,Q8_TOTAL_STOCKHOLDERS_EQUITY,Q8_NET_INCOME,Q8_GROSS_PROFIT,Q8_COST_OF_REVENUES,Q8_REVENUES,Q8_OPERATING_INCOME,Q8_OPERATING_EXPENSES,Q8_EBITDA,Q8_DEPRECIATION_AND_AMORTIZATION,Q8_fiscal_year_end,Q9_TOTAL_ASSETS,Q9_TOTAL_CURRENT_ASSETS,Q9_TOTAL_NONCURRENT_ASSETS,Q9_TOTAL_LIABILITIES,Q9_TOTAL_CURRENT_LIABILITIES,Q9_TOTAL_NONCURRENT_LIABILITIES,Q9_TOTAL_LIABILITIES_AND_EQUITY,Q9_TOTAL_STOCKHOLDERS_EQUITY,Q9_NET_INCOME,Q9_GROSS_PROFIT,Q9_COST_OF_REVENUES,Q9_REVENUES,Q9_OPERATING_INCOME,Q9_OPERATING_EXPENSES,Q9_EBITDA,Q9_DEPRECIATION_AND_AMORTIZATION,Q9_fiscal_year_end,Q10_TOTAL_ASSETS,Q10_TOTAL_CURRENT_ASSETS,Q10_TOTAL_NONCURREN

,fullTimeEmployees,auditRisk,boardRisk,compensationRisk,shareHolderRightsRisk,overallRisk,floatShares,sharesOutstanding,trailingEps,forwardEps,targetHighPrice,targetLowPrice,targetMeanPrice,targetMedianPrice,recommendationMean,numberOfAnalystOpinions,totalCash,totalCashPerShare,ebitda,totalDebt,totalRevenue,revenuePerShare,freeCashflow,operatingCashflow,revenueGrowth,Q0_fiscal_year_end,Q1_TOTAL_ASSETS,Q1_TOTAL_CURRENT_ASSETS,Q1_TOTAL_NONCURRENT_ASSETS,Q1_TOTAL_LIABILITIES,Q1_TOTAL_CURRENT_LIABILITIES,Q1_TOTAL_NONCURRENT_LIABILITIES,Q1_TOTAL_LIABILITIES_AND_EQUITY,Q1_TOTAL_STOCKHOLDERS_EQUITY,Q1_NET_INCOME,Q1_GROSS_PROFIT,Q1_COST_OF_REVENUES,Q1_REVENUES,Q1_OPERATING_INCOME,Q1_OPERATING_EXPENSES,Q1_EBITDA,Q1_DEPRECIATION_AND_AMORTIZATION,Q1_fiscal_year_end,Q2_TOTAL_ASSETS,Q2_TOTAL_CURRENT_ASSETS,Q2_TOTAL_NONCURRENT_ASSETS,Q2_TOTAL_LIABILITIES,Q2_TOTAL_CURRENT_LIABILITIES,Q2_TOTAL_NONCURRENT_LIABILITIES,Q2_TOTAL_LIABILITIES_AND_EQUITY,Q2_TOTAL_STOCKHOLDERS_EQUITY,Q2_NET_INCOME,Q2_GROSS_PROFIT,Q2_COST_OF_REVENUES,Q2_REVENUES,Q2_OPERATING_INCOME,Q2_OPERATING_EXPENSES,Q2_EBITDA,Q2_DEPRECIATION_AND_AMORTIZATION,Q2_fiscal_year_end,Q3_TOTAL_ASSETS,Q3_TOTAL_CURRENT_ASSETS,Q3_TOTAL_NONCURRENT_ASSETS,Q3_TOTAL_LIABILITIES,Q3_TOTAL_CURRENT_LIABILITIES,Q3_TOTAL_NONCURRENT_LIABILITIES,Q3_TOTAL_LIABILITIES_AND_EQUITY,Q3_TOTAL_STOCKHOLDERS_EQUITY,Q3_NET_INCOME,Q3_GROSS_PROFIT,Q3_COST_OF_REVENUES,Q3_REVENUES,Q3_OPERATING_INCOME,Q3_OPERATING_EXPENSES,Q3_EBITDA,Q3_DEPRECIATION_AND_AMORTIZATION,Q3_fiscal_year_end,Q4_TOTAL_ASSETS,Q4_TOTAL_CURRENT_ASSETS,Q4_TOTAL_NONCURRENT_ASSETS,Q4_TOTAL_LIABILITIES,Q4_TOTAL_CURRENT_LIABILITIES,Q4_TOTAL_NONCURRENT_LIABILITIES,Q4_TOTAL_LIABILITIES_AND_EQUITY,Q4_TOTAL_STOCKHOLDERS_EQUITY,Q4_NET_INCOME,Q4_GROSS_PROFIT,Q4_COST_OF_REVENUES,Q4_REVENUES,Q4_OPERATING_INCOME,Q4_OPERATING_EXPENSES,Q4_EBITDA,Q4_DEPRECIATION_AND_AMORTIZATION,Q4_fiscal_year_end,Q5_TOTAL_ASSETS,Q5_TOTAL_CURRENT_ASSETS,Q5_TOTAL_NONCURRENT_ASSETS,Q5_TOTAL_LIABILITIES,Q5_TOTAL_CURRENT_LIABILITIES,Q5_TOTAL_NONCURRENT_LIABILITIES,Q5_TOTAL_LIABILITIES_AND_EQUITY,Q5_TOTAL_STOCKHOLDERS_EQUITY,Q5_NET_INCOME,Q5_GROSS_PROFIT,Q5_COST_OF_REVENUES,Q5_REVENUES,Q5_OPERATING_INCOME,Q5_OPERATING_EXPENSES,Q5_EBITDA,Q5_DEPRECIATION_AND_AMORTIZATION,Q5_fiscal_year_end,Q6_TOTAL_ASSETS,Q6_TOTAL_CURRENT_ASSETS,Q6_TOTAL_NONCURRENT_ASSETS,Q6_TOTAL_LIABILITIES,Q6_TOTAL_CURRENT_LIABILITIES,Q6_TOTAL_NONCURRENT_LIABILITIES,Q6_TOTAL_LIABILITIES_AND_EQUITY,Q6_TOTAL_STOCKHOLDERS_EQUITY,Q6_NET_INCOME,Q6_GROSS_PROFIT,Q6_COST_OF_REVENUES,Q6_REVENUES,Q6_OPERATING_INCOME,Q6_OPERATING_EXPENSES,Q6_EBITDA,Q6_DEPRECIATION_AND_AMORTIZATION,Q6_fiscal_year_end,Q7_TOTAL_ASSETS,Q7_TOTAL_CURRENT_ASSETS,Q7_TOTAL_NONCURRENT_ASSETS,Q7_TOTAL_LIABILITIES,Q7_TOTAL_CURRENT_LIABILITIES,Q7_TOTAL_NONCURRENT_LIABILITIES,Q7_TOTAL_LIABILITIES_AND_EQUITY,Q7_TOTAL_STOCKHOLDERS_EQUITY,Q7_NET_INCOME,Q7_GROSS_PROFIT,Q7_COST_OF_REVENUES,Q7_REVENUES,Q7_OPERATING_INCOME,Q7_OPERATING_EXPENSES,Q7_EBITDA,Q7_DEPRECIATION_AND_AMORTIZATION,Q7_fiscal_year_end,Q8_TOTAL_ASSETS,Q8_TOTAL_CURRENT_ASSETS,Q8_TOTAL_NONCURRENT_ASSETS,Q8_TOTAL_LIABILITIES,Q8_TOTAL_CURRENT_LIABILITIES,Q8_TOTAL_NONCURRENT_LIABILITIES,Q8_TOTAL_LIABILITIES_AND_EQUITY,Q8_TOTAL_STOCKHOLDERS_EQUITY,Q8_NET_INCOME,Q8_GROSS_PROFIT,Q8_COST_OF_REVENUES,Q8_REVENUES,Q8_OPERATING_INCOME,Q8_OPERATING_EXPENSES,Q8_EBITDA,Q8_DEPRECIATION_AND_AMORTIZATION,Q8_fiscal_year_end,Q9_TOTAL_ASSETS,Q9_TOTAL_CURRENT_ASSETS,Q9_TOTAL_NONCURRENT_ASSETS,Q9_TOTAL_LIABILITIES,Q9_TOTAL_CURRENT_LIABILITIES,Q9_TOTAL_NONCURRENT_LIABILITIES,Q9_TOTAL_LIABILITIES_AND_EQUITY,Q9_TOTAL_STOCKHOLDERS_EQUITY,Q9_NET_INCOME,Q9_GROSS_PROFIT,Q9_COST_OF_REVENUES,Q9_REVENUES,Q9_OPERATING_INCOME,Q9_OPERATING_EXPENSES,Q9_EBITDA,Q9_DEPRECIATION_AND_AMORTIZATION,Q9_fiscal_year_end,Q10_TOTAL_ASSETS,Q10_TOTAL_CURRENT_ASSETS,Q10_TOTAL_NONCURRENT_ASSETS,Q10_TOTAL_LIABILITIES,Q10_TOTAL_CURRENT_LIABILITIES,Q10_TOTAL_NONCURRENT_LIABILITIES,Q10_TOTAL_LIABILITIES_AND_EQUITY,Q10_TOTAL_STOCKHOLDERS_EQUITY,Q10_NET_I

In [3]:
# define the targets and categorical variables

TARGETS = [
    "Q0_TOTAL_ASSETS",
    "Q0_TOTAL_LIABILITIES",
    "Q0_TOTAL_STOCKHOLDERS_EQUITY",
    "Q0_GROSS_PROFIT",
    "Q0_COST_OF_REVENUES",
    "Q0_REVENUES",
    "Q0_OPERATING_INCOME",
    "Q0_OPERATING_EXPENSES",
    "Q0_EBITDA",
]

ID_COLUMN = "Id"

CATEGORICAL_COLUMNS = [
    "industry",
    "sector",
    "financialCurrency",
    "recommendationKey",
]

# sanity check
print("Training data types:")
display(train.dtypes.value_counts().to_frame("count"))

print("Test data types:")
display(test.dtypes.value_counts().to_frame("count"))

print("Missing values in training data:", train.isna().sum().sum())
print("Missing values in test data:    ", test.isna().sum().sum())

print("Duplicate training rows:", train.duplicated().sum())
print("Duplicate test rows:    ", test.duplicated().sum())

print("Unique training IDs:", train["Id"].nunique())
print("Unique test IDs:    ", test["Id"].nunique())

# The targets should appear only in the training data
assert all(target in train.columns for target in TARGETS)
assert all(target not in test.columns for target in TARGETS)

train_predictors = set(train.columns) - set(TARGETS)
test_predictors = set(test.columns)

assert train_predictors == test_predictors

print("Train and test predictor columns match.")

categorical_summary = pd.DataFrame({
    "train_unique": train[CATEGORICAL_COLUMNS].nunique(),
    "test_unique": test[CATEGORICAL_COLUMNS].nunique(),
    "train_missing": train[CATEGORICAL_COLUMNS].isna().sum(),
    "test_missing": test[CATEGORICAL_COLUMNS].isna().sum(),
})

display(categorical_summary)

target_summary = pd.DataFrame({
    "mean": train[TARGETS].mean(),
    "median": train[TARGETS].median(),
    "std": train[TARGETS].std(),
    "min": train[TARGETS].min(),
    "max": train[TARGETS].max(),
    "negative_percent": (train[TARGETS] < 0).mean() * 100,
    "zero_percent": (train[TARGETS] == 0).mean() * 100,
})

display(target_summary)

# replace infinite numerical values with missing values

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

# clean categorical columns

for column in CATEGORICAL_COLUMNS:
    train[column] = (
        train[column]
        .astype("string")
        .str.strip()
        .fillna("Missing")
    )

    test[column] = (
        test[column]
        .astype("string")
        .str.strip()
        .fillna("Missing")
    )

print("Missing values after basic cleaning:")
print("Train:", train.isna().sum().sum())
print("Test: ", test.isna().sum().sum())

Training data types:


,count
float64,207
str,4
int64,1


Test data types:


,count
float64,198
str,4
int64,1


Missing values in training data: 0
Missing values in test data:     0
Duplicate training rows: 0
Duplicate test rows:     0
Unique training IDs: 100000
Unique test IDs:     50000
Train and test predictor columns match.


,train_unique,test_unique,train_missing,test_missing
industry,114,114,0,0
sector,10,10,0,0
financialCurrency,3,3,0,0
recommendationKey,7,7,0,0


,mean,median,std,min,max,negative_percent,zero_percent
Q0_TOTAL_ASSETS,"5,284,814,444.9478","823,114,176.7190","29,852,321,132.5600","-218,228,039,147.0449","385,222,182,619.4560",1.6310,0.0000
Q0_TOTAL_LIABILITIES,"2,352,212,315.9907","298,292,234.7479","10,733,070,691.8697","-52,911,235,273.1424","171,434,826,254.6991",1.6300,0.0000
Q0_TOTAL_STOCKHOLDERS_EQUITY,"2,932,602,128.9571","517,793,867.3440","19,965,117,919.9076","-170,079,580,681.3380","232,934,241,636.5500",2.5180,0.0000
Q0_GROSS_PROFIT,"269,263,332.7450","36,202,766.6313","816,622,903.0605","-1,908,190,344.1017","15,217,976,948.8608",3.0870,0.0000
Q0_COST_OF_REVENUES,"463,512,879.2799","38,001,009.4766","1,678,192,459.9819","-2,115,329,946.7986","30,856,872,866.6089",0.6960,0.0000
Q0_REVENUES,"732,786,983.0507","83,324,932.0804","2,256,151,646.8812","34,219.6354","34,393,659,564.1580",0.0000,0.0000
Q0_OPERATING_INCOME,"-61,384,548.7317","-13,362,900.3176","471,515,406.0986","-16,603,043,069.4709","5,088,807,244.6948",79.5420,0.0000
Q0_OPERATING_EXPENSES,"330,647,881.4768","69,434,137.9764","1,023,916,966.9908","440,481.9663","20,859,578,383.4862",0.0000,0.0000
Q0_EBITDA,"-65,470,294.1092","-12,267,757.5517","698,623,428.2093","-25,504,421,742.4117","11,069,418,997.2932",78.0180,0.0000


Missing values after basic cleaning:
Train: 0
Test:  0


### Given the extreme target scales and outliers, using a transformation from the start is theoretically more stable.

In [4]:
# prepare modelling data
test_ids = test[ID_COLUMN].copy()

X = train.drop(columns=TARGETS + [ID_COLUMN])
y = train[TARGETS].copy()

X_test = test.drop(columns=[ID_COLUMN])

# keep the exact same predictor order

X_test = X_test[X.columns]

assert list(X.columns) == list(X_test.columns)
assert ID_COLUMN not in X.columns
assert not set(TARGETS).intersection(X.columns)

print("X shape:     ", X.shape)
print("y shape:     ", y.shape)
print("X_test shape:", X_test.shape)

print("\nNumerical predictors:", X.select_dtypes(include=np.number).shape[1])
print(
    "Categorical predictors:",
    X.select_dtypes(exclude=np.number).shape[1]
)

# transform target variables
def signed_log1p(values):
    """Apply a signed log transformation."""
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.log1p(np.abs(values))


def inverse_signed_log1p(values):
    """Reverse the signed log transformation."""
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.expm1(np.abs(values))


# create transformed targets for model training
# Keep y unchanged for baseline calculation and sMAPE evaluation

y_transformed = pd.DataFrame(
    signed_log1p(y),
    columns=TARGETS,
    index=y.index
)

# confirm that the transformation is reversible
assert np.allclose(
    y.to_numpy(),
    inverse_signed_log1p(y_transformed.to_numpy())
)

print("Target transformation completed.")
display(y_transformed.head())

# define the sMAPE metric

def smape(y_true, y_pred):
    """
    Calculate symmetric mean absolute percentage error.

    Returns sMAPE on a 0–200 scale.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    numerator = 2 * np.abs(y_true - y_pred)
    denominator = np.abs(y_true) + np.abs(y_pred)

    individual_errors = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=float),
        where=denominator != 0
    )

    return 100 * np.mean(individual_errors)


def evaluate_predictions(y_true, y_pred, targets=TARGETS):
    """
    Calculate sMAPE for every target and the average across targets.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    target_scores = {
        target: smape(y_true[:, i], y_pred[:, i])
        for i, target in enumerate(targets)
    }

    target_scores["AVERAGE"] = np.mean(list(target_scores.values()))

    return target_scores

# create five folds CV
cv = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

# save the splits as a list so every model uses the exact same folds
CV_SPLITS = list(cv.split(X))

print(f"Created {len(CV_SPLITS)} cross-validation folds.")

for fold, (train_idx, val_idx) in enumerate(CV_SPLITS, start=1):
    print(
        f"Fold {fold}: "
        f"{len(train_idx):,} training observations, "
        f"{len(val_idx):,} validation observations"
    )



X shape:      (100000, 202)
y shape:      (100000, 9)
X_test shape: (50000, 202)

Numerical predictors: 198
Categorical predictors: 4
Target transformation completed.


,Q0_TOTAL_ASSETS,Q0_TOTAL_LIABILITIES,Q0_TOTAL_STOCKHOLDERS_EQUITY,Q0_GROSS_PROFIT,Q0_COST_OF_REVENUES,Q0_REVENUES,Q0_OPERATING_INCOME,Q0_OPERATING_EXPENSES,Q0_EBITDA
0,20.7668,19.9837,20.1562,16.6340,17.6718,17.9792,-17.7338,18.0212,-17.9259
1,21.8234,21.2821,20.9510,18.5557,18.8577,19.3968,-14.9748,18.5831,-14.9533
2,18.8407,17.6904,18.4601,15.3344,16.0406,16.4465,-16.6541,16.8909,-16.7515
3,23.4952,22.4199,23.0778,20.9744,19.8830,21.2443,-19.7576,21.2338,-19.6923
4,20.3261,19.1365,19.9632,16.6523,16.7432,17.4103,-17.2105,17.6630,-17.2829


Created 5 cross-validation folds.
Fold 1: 80,000 training observations, 20,000 validation observations
Fold 2: 80,000 training observations, 20,000 validation observations
Fold 3: 80,000 training observations, 20,000 validation observations
Fold 4: 80,000 training observations, 20,000 validation observations
Fold 5: 80,000 training observations, 20,000 validation observations


# Naive baseline model

In [5]:
# mean baseline with five-fold cross-validation
baseline_oof = np.zeros(
    (len(y), len(TARGETS)),
    dtype=float
)

baseline_fold_results = []

for fold, (train_idx, val_idx) in enumerate(CV_SPLITS, start=1):

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # calculate means using only the training portion of this fold
    fold_means = y_train.mean().to_numpy()

    # use those means to predict every validation observation
    val_predictions = np.tile(
        fold_means,
        (len(val_idx), 1)
    )

    # store out-of-fold predictions
    baseline_oof[val_idx] = val_predictions

    # calculate validation scores
    fold_scores = evaluate_predictions(
        y_val.to_numpy(),
        val_predictions
    )

    fold_result = {
        "fold": fold,
        "average_smape": fold_scores["AVERAGE"]
    }

    for target in TARGETS:
        fold_result[target] = fold_scores[target]

    baseline_fold_results.append(fold_result)

    print(
        f"Fold {fold}/{N_FOLDS} | "
        f"Validation average sMAPE: "
        f"{fold_scores['AVERAGE']:.4f}"
    )

# Convert the fold results into a table
baseline_fold_results = pd.DataFrame(
    baseline_fold_results
)

# Calculate the overall out-of-fold sMAPE
baseline_oof_scores = evaluate_predictions(
    y.to_numpy(),
    baseline_oof
)

# Report the mean and standard deviation across folds
baseline_cv_mean = baseline_fold_results[
    "average_smape"
].mean()

baseline_cv_std = baseline_fold_results[
    "average_smape"
].std()

print(
    f"Baseline mean CV sMAPE: "
    f"{baseline_cv_mean:.4f}"
)

print(
    f"Baseline CV standard deviation: "
    f"{baseline_cv_std:.4f}"
)

print(
    f"Baseline CV sMAPE: "
    f"{baseline_cv_mean:.4f} ± {baseline_cv_std:.4f}"
)

# display target-level baseline results

baseline_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        baseline_oof_scores[target]
        for target in TARGETS
    ] + [baseline_oof_scores["AVERAGE"]]
})

display(baseline_results)

Fold 1/5 | Validation average sMAPE: 136.1500


Fold 2/5 | Validation average sMAPE: 136.2535
Fold 3/5 | Validation average sMAPE: 135.6418
Fold 4/5 | Validation average sMAPE: 135.2469
Fold 5/5 | Validation average sMAPE: 136.3733
Baseline mean CV sMAPE: 135.9331
Baseline CV standard deviation: 0.4741
Baseline CV sMAPE: 135.9331 ± 0.4741


,target,smape
0,Q0_TOTAL_ASSETS,137.2302
1,Q0_TOTAL_LIABILITIES,140.8383
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,134.9773
3,Q0_GROSS_PROFIT,138.0402
4,Q0_COST_OF_REVENUES,144.0142
5,Q0_REVENUES,140.6332
6,Q0_OPERATING_INCOME,128.5692
7,Q0_OPERATING_EXPENSES,127.3349
8,Q0_EBITDA,131.7603
9,AVERAGE,135.9331


# Non-deep-learning Models

## Elastic Net

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import time

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()

# preprocessing fitted separately within each fold
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),
        categorical_columns
    )
])

# hyperparameters to test
elasticnet_configs = [
    {"alpha": alpha, "l1_ratio": l1_ratio}
    for alpha in [0.1, 1.0]
    for l1_ratio in [0.1, 0.9]
]

config_results = {
    i: {
        "params": params,
        "oof_predictions": np.zeros((len(y), len(TARGETS))),
        "fold_scores": []
    }
    for i, params in enumerate(elasticnet_configs)
}


# Train and evaluate every configuration
for fold, (train_idx, val_idx) in enumerate(CV_SPLITS, start=1):

    print(f"\nFold {fold}/{N_FOLDS}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_transformed = y_transformed.iloc[train_idx]
    y_val_original = y.iloc[val_idx]

    # fit preprocessing using only this fold's training data
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    for config_id, result in config_results.items():

        params = result["params"]
        start_time = time.time()

        model = MultiTaskElasticNet(
            alpha=params["alpha"],
            l1_ratio=params["l1_ratio"],
            max_iter=500,
            tol=1e-2,
            selection="random",
            random_state=SEED
        )

        print(
            f"  Training alpha={params['alpha']}, "
            f"l1_ratio={params['l1_ratio']}..."
        )

        model.fit(
            X_train_processed,
            y_train_transformed
        )

        # validation predictions in transformed space
        val_predictions_transformed = model.predict(
            X_val_processed
        )

        # return predictions to original units
        val_predictions = inverse_signed_log1p(
            val_predictions_transformed
        )

        result["oof_predictions"][val_idx] = val_predictions

        fold_score = evaluate_predictions(
            y_val_original.to_numpy(),
            val_predictions
        )["AVERAGE"]

        result["fold_scores"].append(fold_score)

        elapsed = time.time() - start_time

        print(
            f"alpha={params['alpha']}, "
            f"l1_ratio={params['l1_ratio']} | "
            f"sMAPE={fold_score:.4f} | "
            f"iterations={model.n_iter_} | "
            f"time={elapsed:.1f}s"
        )

# Summarize the configurations and select the best one
elasticnet_results = []

for config_id, result in config_results.items():

    fold_scores = result["fold_scores"]

    elasticnet_results.append({
        "config_id": config_id,
        "alpha": result["params"]["alpha"],
        "l1_ratio": result["params"]["l1_ratio"],
        "mean_cv_smape": np.mean(fold_scores),
        "std_cv_smape": np.std(fold_scores, ddof=1)
    })

elasticnet_results = (
    pd.DataFrame(elasticnet_results)
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(elasticnet_results)

best_config_id = int(
    elasticnet_results.loc[0, "config_id"]
)

best_elasticnet = config_results[best_config_id]
best_elasticnet_params = best_elasticnet["params"]
elasticnet_oof = best_elasticnet["oof_predictions"]

elasticnet_scores = evaluate_predictions(
    y.to_numpy(),
    elasticnet_oof
)

elasticnet_cv_mean = elasticnet_results.loc[
    0, "mean_cv_smape"
]

elasticnet_cv_std = elasticnet_results.loc[
    0, "std_cv_smape"
]

print("Best Elastic Net parameters:", best_elasticnet_params)

print(
    f"Elastic Net CV sMAPE: "
    f"{elasticnet_cv_mean:.4f} ± "
    f"{elasticnet_cv_std:.4f}"
)

# display elastic net results
elasticnet_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        elasticnet_scores[target]
        for target in TARGETS
    ] + [elasticnet_scores["AVERAGE"]]
})
display(elasticnet_target_results)


Fold 1/5
  Training alpha=0.1, l1_ratio=0.1...
alpha=0.1, l1_ratio=0.1 | sMAPE=96.5828 | iterations=57 | time=68.0s
  Training alpha=0.1, l1_ratio=0.9...
alpha=0.1, l1_ratio=0.9 | sMAPE=94.7649 | iterations=48 | time=34.0s
  Training alpha=1.0, l1_ratio=0.1...
alpha=1.0, l1_ratio=0.1 | sMAPE=113.0229 | iterations=12 | time=9.5s
  Training alpha=1.0, l1_ratio=0.9...
alpha=1.0, l1_ratio=0.9 | sMAPE=112.5291 | iterations=17 | time=9.5s

Fold 2/5
  Training alpha=0.1, l1_ratio=0.1...
alpha=0.1, l1_ratio=0.1 | sMAPE=96.4858 | iterations=57 | time=62.7s
  Training alpha=0.1, l1_ratio=0.9...
alpha=0.1, l1_ratio=0.9 | sMAPE=94.7057 | iterations=62 | time=47.1s
  Training alpha=1.0, l1_ratio=0.1...
alpha=1.0, l1_ratio=0.1 | sMAPE=113.2490 | iterations=12 | time=9.6s
  Training alpha=1.0, l1_ratio=0.9...
alpha=1.0, l1_ratio=0.9 | sMAPE=112.4447 | iterations=17 | time=9.0s

Fold 3/5
  Training alpha=0.1, l1_ratio=0.1...
alpha=0.1, l1_ratio=0.1 | sMAPE=96.4238 | iterations=57 | time=60.5s
  Train

,config_id,alpha,l1_ratio,mean_cv_smape,std_cv_smape
0,1,0.1000,0.9000,94.6253,0.2182
1,0,0.1000,0.1000,96.4495,0.1805
2,3,1.0000,0.9000,112.3698,0.1386
3,2,1.0000,0.1000,113.0380,0.1507


Best Elastic Net parameters: {'alpha': 0.1, 'l1_ratio': 0.9}
Elastic Net CV sMAPE: 94.6253 ± 0.2182


,target,smape
0,Q0_TOTAL_ASSETS,69.4578
1,Q0_TOTAL_LIABILITIES,68.5450
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,75.2029
3,Q0_GROSS_PROFIT,97.4697
4,Q0_COST_OF_REVENUES,77.1059
5,Q0_REVENUES,69.0335
6,Q0_OPERATING_INCOME,166.2764
7,Q0_OPERATING_EXPENSES,59.1030
8,Q0_EBITDA,169.4336
9,AVERAGE,94.6253


The best Elastic Net configuration is $\alpha=0.1$ and l1_ratio = 0.9, with an average cross-validation sMAPE of about $94.63$. It uses relatively weak regularization overall, with the penalty dominated by the L1 component. This makes the model behave more like Lasso than Ridge, encouraging sparsity while retaining a small amount of L2 regularization.

Performance varies substantially across targets. The lowest sMAPE values are for Q0_OPERATING_EXPENSES at about $59.10$, Q0_TOTAL_LIABILITIES at about $68.55$, Q0_REVENUES at about $69.03$, and Q0_TOTAL_ASSETS at about $69.46$

The weakest performance is for Q0_EBITDA at about $169.43$ and Q0_OPERATING_INCOME at about $166.28$

Overall, the model performs much better on balance-sheet and revenue-related targets than on operating-profitability targets such as EBITDA and operating income

## Random Forest

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
import time

# preprocess numerical and categorical predictors
rf_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_columns
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                )
            )
        ]),
        categorical_columns
    )
])

# define hyperparameter configurations
rf_configs = [
    {
        "n_estimators": 100,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 200,
        "max_depth": 25,
        "min_samples_leaf": 2,
        "max_features": 0.5
    }
]

# initialize results
rf_config_results = {
    i: {
        "params": params,
        "oof_predictions": np.zeros((len(y), len(TARGETS))),
        "fold_scores": []
    }
    for i, params in enumerate(rf_configs)
}

# train and evaluate each configuration
for fold, (train_idx, val_idx) in enumerate(CV_SPLITS, start=1):

    print(f"\nFold {fold}/{N_FOLDS}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_transformed = y_transformed.iloc[train_idx]
    y_val_original = y.iloc[val_idx]

    # fit preprocessing on the training fold
    X_train_processed = rf_preprocessor.fit_transform(X_train)
    X_val_processed = rf_preprocessor.transform(X_val)

    for config_id, result in rf_config_results.items():

        params = result["params"]
        start_time = time.time()

        print(
            f"  Training {params['n_estimators']} trees, "
            f"max_depth={params['max_depth']}..."
        )

        model = RandomForestRegressor(
            **params,
            n_jobs=-1,
            random_state=SEED
        )

        model.fit(
            X_train_processed,
            y_train_transformed
        )

        # predict and return to original units
        val_predictions_transformed = model.predict(
            X_val_processed
        )

        val_predictions = inverse_signed_log1p(
            val_predictions_transformed
        )

        # store out-of-fold predictions
        result["oof_predictions"][val_idx] = val_predictions

        # calculate validation sMAPE
        fold_score = evaluate_predictions(
            y_val_original.to_numpy(),
            val_predictions
        )["AVERAGE"]

        result["fold_scores"].append(fold_score)

        elapsed = time.time() - start_time

        print(
            f"  sMAPE={fold_score:.4f} | "
            f"time={elapsed:.1f}s"
        )

# summarize configurations
rf_results = []

for config_id, result in rf_config_results.items():

    rf_results.append({
        "config_id": config_id,
        **result["params"],
        "mean_cv_smape": np.mean(result["fold_scores"]),
        "std_cv_smape": np.std(
            result["fold_scores"],
            ddof=1
        )
    })

rf_results = (
    pd.DataFrame(rf_results)
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(rf_results)

# select the best configuration
best_rf_config_id = int(
    rf_results.loc[0, "config_id"]
)

best_rf = rf_config_results[best_rf_config_id]
best_rf_params = best_rf["params"]
rf_oof = best_rf["oof_predictions"]

# calculate target-level results
rf_scores = evaluate_predictions(
    y.to_numpy(),
    rf_oof
)

rf_cv_mean = rf_results.loc[0, "mean_cv_smape"]
rf_cv_std = rf_results.loc[0, "std_cv_smape"]

print("Best Random Forest parameters:", best_rf_params)

print(
    f"Random Forest CV sMAPE: "
    f"{rf_cv_mean:.4f} ± {rf_cv_std:.4f}"
)

# display target-level results
rf_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        rf_scores[target]
        for target in TARGETS
    ] + [rf_scores["AVERAGE"]]
})

display(rf_target_results)


Fold 1/5
  Training 100 trees, max_depth=15...
  sMAPE=36.5021 | time=33.3s
  Training 200 trees, max_depth=25...
  sMAPE=27.4731 | time=836.9s

Fold 2/5
  Training 100 trees, max_depth=15...
  sMAPE=36.6853 | time=44.6s
  Training 200 trees, max_depth=25...
  sMAPE=27.3335 | time=883.6s

Fold 3/5
  Training 100 trees, max_depth=15...
  sMAPE=36.3860 | time=45.1s
  Training 200 trees, max_depth=25...
  sMAPE=27.2944 | time=863.5s

Fold 4/5
  Training 100 trees, max_depth=15...
  sMAPE=36.9083 | time=27.5s
  Training 200 trees, max_depth=25...
  sMAPE=27.5721 | time=524.8s

Fold 5/5
  Training 100 trees, max_depth=15...
  sMAPE=36.3279 | time=23.4s
  Training 200 trees, max_depth=25...
  sMAPE=27.2798 | time=481.6s


,config_id,n_estimators,max_depth,min_samples_leaf,max_features,mean_cv_smape,std_cv_smape
0,1,200,25,2,0.5000,27.3906,0.1270
1,0,100,15,5,sqrt,36.5619,0.2369


Best Random Forest parameters: {'n_estimators': 200, 'max_depth': 25, 'min_samples_leaf': 2, 'max_features': 0.5}
Random Forest CV sMAPE: 27.3906 ± 0.1270


,target,smape
0,Q0_TOTAL_ASSETS,7.8038
1,Q0_TOTAL_LIABILITIES,12.1654
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,13.4998
3,Q0_GROSS_PROFIT,26.6596
4,Q0_COST_OF_REVENUES,16.0657
5,Q0_REVENUES,10.2523
6,Q0_OPERATING_INCOME,75.9933
7,Q0_OPERATING_EXPENSES,7.7518
8,Q0_EBITDA,76.3236
9,AVERAGE,27.3906


Two Random Forest configurations were evaluated using five-fold cross-validation. All models were trained on signed-log-transformed targets, while sMAPE was calculated after converting predictions back to their original units. The stronger configuration used 200 trees, a maximum depth of 25, a minimum leaf size of 2, and 50% of predictors at each split. It achieved a mean CV sMAPE of 27.3906 ± 0.1270, compared with 36.5619 ± 0.2369 for the simpler configuration and 135.9331 for the mean baseline. The small standard deviation indicates stable performance across folds. Random Forest performed particularly well for assets, liabilities, revenues, and operating expenses, but had substantially higher errors for operating income and EBITDA.

## XGBoost

In [ ]:
import xgboost as xgb
import time

# preprocess numerical and categorical predictors
xgb_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=True,
                        dtype=np.float32
                    )
                )
            ]),
            categorical_columns
        )
    ],
    sparse_threshold=1.0
)

# define hyperparameter configurations
xgb_configs = [
    {
        "learning_rate": 0.05,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0
    },
    {
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 5.0
    },
    {
        "learning_rate": 0.03,
        "max_depth": 8,
        "min_child_weight": 10,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 10.0
    },
    {
        "learning_rate": 0.03,
        "max_depth": 5,
        "min_child_weight": 1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0
    }
]

# initialize results
xgb_config_results = {
    i: {
        "params": params,
        "oof_predictions": np.zeros(
            (len(y), len(TARGETS))
        ),
        "fold_scores": [],
        "best_iterations": []
    }
    for i, params in enumerate(xgb_configs, start=1)
}

# train and evaluate each configuration
for fold, (train_idx, val_idx) in enumerate(
    CV_SPLITS,
    start=1
):

    print(f"\nFold {fold}/{N_FOLDS}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_transformed = y_transformed.iloc[train_idx]
    y_val_transformed = y_transformed.iloc[val_idx]
    y_val_original = y.iloc[val_idx]

    # fit preprocessing on the training fold
    X_train_processed = xgb_preprocessor.fit_transform(
        X_train
    )

    X_val_processed = xgb_preprocessor.transform(
        X_val
    )

    for config_id, result in xgb_config_results.items():

        params = result["params"]
        start_time = time.time()

        print(
            f"\n  Config {config_id}/{len(xgb_configs)} | "
            f"learning_rate={params['learning_rate']} | "
            f"max_depth={params['max_depth']}"
        )

        fold_predictions = np.zeros(
            (len(val_idx), len(TARGETS))
        )

        fold_best_iterations = []

        # train one model per target
        for target_id, target in enumerate(TARGETS):

            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                n_estimators=3000,
                learning_rate=params["learning_rate"],
                max_depth=params["max_depth"],
                min_child_weight=params["min_child_weight"],
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                reg_alpha=params["reg_alpha"],
                reg_lambda=params["reg_lambda"],
                tree_method="hist",
                max_bin=256,
                eval_metric="mae",
                early_stopping_rounds=100,
                random_state=SEED,
                n_jobs=-1,
                verbosity=0
            )

            model.fit(
                X_train_processed,
                y_train_transformed[target],
                eval_set=[
                    (
                        X_val_processed,
                        y_val_transformed[target]
                    )
                ],
                verbose=False
            )

            # predict and return to original units
            target_predictions_transformed = model.predict(
                X_val_processed
            )

            target_predictions = inverse_signed_log1p(
                target_predictions_transformed
            )

            fold_predictions[
                :,
                target_id
            ] = target_predictions

            target_score = smape(
                y_val_original[target],
                target_predictions
            )

            best_iteration = model.best_iteration
            fold_best_iterations.append(best_iteration)

            print(
                f"    {target} | "
                f"sMAPE={target_score:.4f} | "
                f"best iteration={best_iteration}"
            )

        # store out-of-fold predictions
        result["oof_predictions"][
            val_idx
        ] = fold_predictions

        # store average best iteration
        result["best_iterations"].append(
            np.mean(fold_best_iterations)
        )

        # calculate fold sMAPE
        fold_score = evaluate_predictions(
            y_val_original.to_numpy(),
            fold_predictions
        )["AVERAGE"]

        result["fold_scores"].append(fold_score)

        elapsed = time.time() - start_time

        print(
            f"  Fold average sMAPE={fold_score:.4f} | "
            f"time={elapsed / 60:.1f} minutes"
        )

# summarize configurations
xgb_results = []

for config_id, result in xgb_config_results.items():

    xgb_results.append({
        "config_id": config_id,
        **result["params"],
        "mean_cv_smape": np.mean(
            result["fold_scores"]
        ),
        "std_cv_smape": np.std(
            result["fold_scores"],
            ddof=1
        ),
        "mean_best_iteration": np.mean(
            result["best_iterations"]
        )
    })

xgb_results = (
    pd.DataFrame(xgb_results)
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(xgb_results)

# summarize target-level results for every configuration
xgb_config_target_results = []

for config_id, result in xgb_config_results.items():

    config_scores = evaluate_predictions(
        y.to_numpy(),
        result["oof_predictions"]
    )

    for target in TARGETS + ["AVERAGE"]:
        xgb_config_target_results.append({
            "config_id": config_id,
            "target": target,
            "smape": config_scores[target]
        })

xgb_config_target_results = pd.DataFrame(
    xgb_config_target_results
)

display(xgb_config_target_results)

# select the best configuration
best_xgb_config_id = int(
    xgb_results.loc[0, "config_id"]
)

best_xgb = xgb_config_results[
    best_xgb_config_id
]

best_xgb_params = best_xgb["params"]
xgb_oof = best_xgb["oof_predictions"]

# calculate target-level results
xgb_scores = evaluate_predictions(
    y.to_numpy(),
    xgb_oof
)

xgb_cv_mean = xgb_results.loc[
    0,
    "mean_cv_smape"
]

xgb_cv_std = xgb_results.loc[
    0,
    "std_cv_smape"
]

print("Best XGBoost parameters:", best_xgb_params)

print(
    f"XGBoost CV sMAPE: "
    f"{xgb_cv_mean:.4f} ± {xgb_cv_std:.4f}"
)

# display target-level results
xgb_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        xgb_scores[target]
        for target in TARGETS
    ] + [xgb_scores["AVERAGE"]]
})

display(xgb_target_results)


Fold 1/5

  Config 2/4 | learning_rate=0.05 | max_depth=4
    Q0_TOTAL_ASSETS | sMAPE=3.0515 | best iteration=2999
    Q0_TOTAL_LIABILITIES | sMAPE=4.3585 | best iteration=2999
    Q0_TOTAL_STOCKHOLDERS_EQUITY | sMAPE=11.2026 | best iteration=2999
    Q0_GROSS_PROFIT | sMAPE=32.0185 | best iteration=2997
    Q0_COST_OF_REVENUES | sMAPE=9.3778 | best iteration=2999
    Q0_REVENUES | sMAPE=5.5049 | best iteration=2999
    Q0_OPERATING_INCOME | sMAPE=87.5703 | best iteration=2255
    Q0_OPERATING_EXPENSES | sMAPE=2.8141 | best iteration=2999
    Q0_EBITDA | sMAPE=90.1123 | best iteration=2961
  Fold average sMAPE=27.3345 | time=24.8 minutes

  Config 3/4 | learning_rate=0.05 | max_depth=6
    Q0_TOTAL_ASSETS | sMAPE=2.1127 | best iteration=2999
    Q0_TOTAL_LIABILITIES | sMAPE=3.6743 | best iteration=2999
    Q0_TOTAL_STOCKHOLDERS_EQUITY | sMAPE=8.0272 | best iteration=2999
    Q0_GROSS_PROFIT | sMAPE=28.6269 | best iteration=804
    Q0_COST_OF_REVENUES | sMAPE=7.5189 | best iteration=29

,config_id,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,mean_cv_smape,std_cv_smape,mean_best_iteration
0,3,0.0300,8,10,0.8000,0.8000,0.1000,10.0000,23.2273,0.1455,"2,320.8444"
1,2,0.0500,6,5,0.8000,0.8000,0.1000,5.0000,24.4349,0.0914,"2,424.3111"
2,4,0.0300,5,1,1.0000,1.0000,0.0000,1.0000,25.3248,0.1277,"2,992.7778"
3,1,0.0500,4,5,0.8000,0.8000,0.0000,1.0000,27.2705,0.1084,"2,898.9556"


,config_id,target,smape
0,1,Q0_TOTAL_ASSETS,3.0968
1,1,Q0_TOTAL_LIABILITIES,4.2898
2,1,Q0_TOTAL_STOCKHOLDERS_EQUITY,11.2083
3,1,Q0_GROSS_PROFIT,32.3033
4,1,Q0_COST_OF_REVENUES,9.3542
5,1,Q0_REVENUES,5.5013
6,1,Q0_OPERATING_INCOME,86.9177
7,1,Q0_OPERATING_EXPENSES,2.8339
8,1,Q0_EBITDA,89.9295
9,1,AVERAGE,27.2705


Best XGBoost parameters: {'learning_rate': 0.03, 'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 10.0}
XGBoost CV sMAPE: 23.2273 ± 0.1455


,target,smape
0,Q0_TOTAL_ASSETS,2.2363
1,Q0_TOTAL_LIABILITIES,3.7211
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,7.6258
3,Q0_GROSS_PROFIT,26.5696
4,Q0_COST_OF_REVENUES,7.5704
5,Q0_REVENUES,5.0352
6,Q0_OPERATING_INCOME,77.1535
7,Q0_OPERATING_EXPENSES,2.0420
8,Q0_EBITDA,77.0919
9,AVERAGE,23.2273


Four XGBoost configurations were evaluated using five-fold cross-validation. A separate model was trained for each target using the signed-log-transformed values, with early stopping used to determine the appropriate number of boosting iterations. Predictions were converted back to their original units before calculating sMAPE.

Configuration 3 produced the best overall performance, with a mean CV sMAPE of 23.2273 ± 0.1455. It used a learning rate of 0.03, maximum depth of 8, minimum child weight of 10, 80% row and feature sampling, and relatively strong regularization. Its average best iteration was approximately 2,321, indicating that the 3,000-tree limit was sufficient. The low standard deviation also suggests that its performance was stable across the five folds.

XGBoost improved upon the Random Forest result of 27.3906 by approximately 15.2% and substantially outperformed the mean baseline of 135.9331. It performed especially well when predicting assets, liabilities, revenues, and operating expenses. However, operating income and EBITDA remained difficult, with sMAPEs of 77.1535 and 77.0919, respectively. Random Forest performed slightly better for these two targets, suggesting that a later target-level combination could use XGBoost for the other seven targets and Random Forest for operating income and EBITDA. Based on the out-of-fold results, this combination would achieve an estimated sMAPE of approximately 23.01. But we are still going to try out other models to get more information about next steps. 

### Accounting-Based Post-Processing

The financial targets are not completely independent. Several are connected through accounting relationships, but the models predict each target separately and therefore do not automatically enforce these relationships. The signed-log transformation also makes additive relationships more difficult for the models to learn directly.

Inspection of the training data showed that operating income equals gross profit minus operating expenses for every observation. Gross profit is also closely related to revenue minus cost of revenues. Because Q0 depreciation and amortization is unavailable, Q1 depreciation and amortization was used as the most recent available approximation when deriving EBITDA.

The following post-processing rules were evaluated:

- Gross profit = predicted revenue − predicted cost of revenues
- Operating income = derived gross profit − predicted operating expenses
- EBITDA = derived operating income + Q1 depreciation and amortization

These adjustments use only out-of-fold predictions and predictors available in the test data, so they do not introduce data leakage. Both the original and adjusted predictions were evaluated using the same five cross-validation folds, and the adjustments were retained only when they improved sMAPE.

In [13]:
# define target positions
revenue_id = TARGETS.index("Q0_REVENUES")
cost_id = TARGETS.index("Q0_COST_OF_REVENUES")
gross_profit_id = TARGETS.index("Q0_GROSS_PROFIT")
operating_expenses_id = TARGETS.index(
    "Q0_OPERATING_EXPENSES"
)
operating_income_id = TARGETS.index(
    "Q0_OPERATING_INCOME"
)
ebitda_id = TARGETS.index("Q0_EBITDA")


# apply accounting relationships
def apply_accounting_adjustments(oof_predictions):

    adjusted = oof_predictions.copy()

    # derive gross profit
    adjusted[:, gross_profit_id] = (
        adjusted[:, revenue_id] -
        adjusted[:, cost_id]
    )

    # derive operating income
    adjusted[:, operating_income_id] = (
        adjusted[:, gross_profit_id] -
        adjusted[:, operating_expenses_id]
    )

    # derive EBITDA
    adjusted[:, ebitda_id] = (
        adjusted[:, operating_income_id] +
        X["Q1_DEPRECIATION_AND_AMORTIZATION"].to_numpy()
    )

    return adjusted


# store original model predictions
original_model_predictions = {
    "Elastic Net": elasticnet_oof,
    "Random Forest": rf_oof,
    "XGBoost": xgb_oof
}


# apply accounting adjustments
accounting_model_predictions = {
    model_name: apply_accounting_adjustments(predictions)
    for model_name, predictions
    in original_model_predictions.items()
}


# keep individual adjusted predictions
elasticnet_accounting_oof = (
    accounting_model_predictions["Elastic Net"]
)

rf_accounting_oof = (
    accounting_model_predictions["Random Forest"]
)

xgb_accounting_oof = (
    accounting_model_predictions["XGBoost"]
)


# calculate model-level results
accounting_summary = []

for model_name in original_model_predictions:

    original_predictions = (
        original_model_predictions[model_name]
    )

    adjusted_predictions = (
        accounting_model_predictions[model_name]
    )

    original_fold_scores = []
    adjusted_fold_scores = []

    for _, val_idx in CV_SPLITS:

        original_fold_scores.append(
            evaluate_predictions(
                y.iloc[val_idx].to_numpy(),
                original_predictions[val_idx]
            )["AVERAGE"]
        )

        adjusted_fold_scores.append(
            evaluate_predictions(
                y.iloc[val_idx].to_numpy(),
                adjusted_predictions[val_idx]
            )["AVERAGE"]
        )

    accounting_summary.append({
        "model": model_name,
        "original_cv_smape": np.mean(
            original_fold_scores
        ),
        "accounting_cv_smape": np.mean(
            adjusted_fold_scores
        ),
        "accounting_cv_std": np.std(
            adjusted_fold_scores,
            ddof=1
        ),
        "improvement": (
            np.mean(original_fold_scores) -
            np.mean(adjusted_fold_scores)
        )
    })


accounting_summary = pd.DataFrame(
    accounting_summary
).sort_values(
    "accounting_cv_smape"
).reset_index(drop=True)

display(accounting_summary)


# calculate affected target results
affected_targets = [
    "Q0_GROSS_PROFIT",
    "Q0_OPERATING_INCOME",
    "Q0_EBITDA"
]

accounting_target_results = []

for model_name in original_model_predictions:

    original_scores = evaluate_predictions(
        y.to_numpy(),
        original_model_predictions[model_name]
    )

    adjusted_scores = evaluate_predictions(
        y.to_numpy(),
        accounting_model_predictions[model_name]
    )

    for target in affected_targets:

        accounting_target_results.append({
            "model": model_name,
            "target": target,
            "original_smape": original_scores[target],
            "accounting_smape": adjusted_scores[target]
        })


accounting_target_results = pd.DataFrame(
    accounting_target_results
)

display(accounting_target_results)

,model,original_cv_smape,accounting_cv_smape,accounting_cv_std,improvement
0,XGBoost,23.2273,14.0320,0.1024,9.1954
1,Random Forest,27.3906,19.6427,0.0452,7.7479
2,Elastic Net,94.6253,76.7453,0.2473,17.8800


,model,target,original_smape,accounting_smape
0,Elastic Net,Q0_GROSS_PROFIT,97.4697,80.4759
1,Elastic Net,Q0_OPERATING_INCOME,166.2764,98.2536
2,Elastic Net,Q0_EBITDA,169.4336,93.5303
3,Random Forest,Q0_GROSS_PROFIT,26.6596,21.2115
4,Random Forest,Q0_OPERATING_INCOME,75.9933,44.8645
5,Random Forest,Q0_EBITDA,76.3236,43.1694
6,XGBoost,Q0_GROSS_PROFIT,26.5696,17.9928
7,XGBoost,Q0_OPERATING_INCOME,77.1535,40.8649
8,XGBoost,Q0_EBITDA,77.0919,39.1992


Accounting-based post-processing improved all three models. XGBoost remained the strongest approach, with its mean CV sMAPE decreasing from 23.2273 to 14.0320. Random Forest improved from 27.3906 to 19.6427, while Elastic Net improved from 94.6253 to 76.7453. The adjusted XGBoost model also remained stable across folds, with a CV standard deviation of 0.1024.

For XGBoost, the gross-profit sMAPE decreased from 26.5696 to 17.9928, operating-income sMAPE decreased from 77.1535 to 40.8649, and EBITDA sMAPE decreased from 77.0919 to 39.1992. These results show that part of the original prediction error came from independently predicting financially related targets without enforcing their accounting relationships.

Although the adjustments substantially improved performance, operating income and EBITDA remain the most difficult targets. The next step is to predict Q0 depreciation and amortization as a separate derived target and then calculate EBITDA using predicted operating income plus predicted Q0 depreciation and amortization. This may improve upon using Q1 depreciation and amortization as a direct proxy. 

## Feature engineer for Q0 D&A Prediction
Q0 depreciation and amortization is not provided directly, so it was calculated in the training data as Q0 EBITDA minus Q0 operating income. Three approaches were compared: *carrying forward Q1 D&A, using Ridge regression with Q1–Q10 D&A values, and using XGBoost with all predictors*. Ridge provides a simple model of historical D&A patterns, while XGBoost can capture nonlinear relationships with other financial variables. Each method was evaluated using the same five cross-validation folds based on the sMAPE of the resulting EBITDA prediction. The best method was then used to calculate EBITDA as predicted operating income plus predicted Q0 D&A.

In [14]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.preprocessing import FunctionTransformer
import time


# create Q0 D&A target
q0_da = (
    y["Q0_EBITDA"] -
    y["Q0_OPERATING_INCOME"]
)

q0_da_transformed = signed_log1p(q0_da)


# define D&A history
da_history_columns = [
    f"Q{quarter}_DEPRECIATION_AND_AMORTIZATION"
    for quarter in range(1, 11)
]


# initialize OOF predictions
ridge_da_oof = np.zeros(len(y))
xgb_da_oof = np.zeros(len(y))

ridge_fold_results = []
xgb_da_fold_results = []
xgb_da_best_iterations = []


# train Ridge and XGBoost
for fold, (train_idx, val_idx) in enumerate(
    CV_SPLITS,
    start=1
):

    print(f"\nFold {fold}/{N_FOLDS}")

    y_train_da = q0_da_transformed[train_idx]
    y_val_da_original = q0_da.iloc[val_idx]

    operating_income_predictions = (
        xgb_accounting_oof[
            val_idx,
            operating_income_id
        ]
    )


    # train Ridge using D&A history
    ridge_start = time.time()

    ridge_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "log",
            FunctionTransformer(
                signed_log1p,
                validate=False
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            Ridge(alpha=1.0)
        )
    ])

    ridge_pipeline.fit(
        X.iloc[train_idx][da_history_columns],
        y_train_da
    )

    ridge_predictions_transformed = (
        ridge_pipeline.predict(
            X.iloc[val_idx][da_history_columns]
        )
    )

    ridge_predictions = inverse_signed_log1p(
        ridge_predictions_transformed
    )

    ridge_da_oof[val_idx] = ridge_predictions

    ridge_da_smape = smape(
        y_val_da_original,
        ridge_predictions
    )

    ridge_ebitda_predictions = (
        operating_income_predictions +
        ridge_predictions
    )

    ridge_ebitda_smape = smape(
        y.iloc[val_idx]["Q0_EBITDA"],
        ridge_ebitda_predictions
    )

    ridge_fold_results.append(
        ridge_ebitda_smape
    )

    print(
        f"  Ridge | "
        f"D&A sMAPE={ridge_da_smape:.4f} | "
        f"EBITDA sMAPE={ridge_ebitda_smape:.4f} | "
        f"time={time.time() - ridge_start:.1f}s"
    )


    # preprocess all predictors for XGBoost
    xgb_start = time.time()

    fold_preprocessor = clone(
        xgb_preprocessor
    )

    X_train_processed = (
        fold_preprocessor.fit_transform(
            X.iloc[train_idx]
        )
    )

    X_val_processed = (
        fold_preprocessor.transform(
            X.iloc[val_idx]
        )
    )


    # train XGBoost using all predictors
    da_xgb_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=10.0,
        tree_method="hist",
        max_bin=256,
        eval_metric="mae",
        early_stopping_rounds=100,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0
    )

    da_xgb_model.fit(
        X_train_processed,
        y_train_da,
        eval_set=[
            (
                X_val_processed,
                q0_da_transformed[val_idx]
            )
        ],
        verbose=False
    )

    xgb_predictions_transformed = (
        da_xgb_model.predict(
            X_val_processed
        )
    )

    xgb_predictions = inverse_signed_log1p(
        xgb_predictions_transformed
    )

    xgb_da_oof[val_idx] = xgb_predictions

    xgb_da_smape = smape(
        y_val_da_original,
        xgb_predictions
    )

    xgb_ebitda_predictions = (
        operating_income_predictions +
        xgb_predictions
    )

    xgb_ebitda_smape = smape(
        y.iloc[val_idx]["Q0_EBITDA"],
        xgb_ebitda_predictions
    )

    xgb_da_fold_results.append(
        xgb_ebitda_smape
    )

    xgb_da_best_iterations.append(
        da_xgb_model.best_iteration
    )

    print(
        f"  XGBoost | "
        f"D&A sMAPE={xgb_da_smape:.4f} | "
        f"EBITDA sMAPE={xgb_ebitda_smape:.4f} | "
        f"best iteration={da_xgb_model.best_iteration} | "
        f"time={(time.time() - xgb_start) / 60:.1f} minutes"
    )


# define D&A prediction methods
da_prediction_methods = {
    "Q1 Carry-Forward": (
        X[
            "Q1_DEPRECIATION_AND_AMORTIZATION"
        ].to_numpy()
    ),
    "Ridge": ridge_da_oof,
    "XGBoost": xgb_da_oof
}


# compare D&A methods
da_comparison = []
da_ebitda_predictions = {}

for method, da_predictions in (
    da_prediction_methods.items()
):

    ebitda_predictions = (
        xgb_accounting_oof[
            :,
            operating_income_id
        ] +
        da_predictions
    )

    da_ebitda_predictions[
        method
    ] = ebitda_predictions

    ebitda_fold_scores = []
    overall_fold_scores = []

    adjusted_predictions = (
        xgb_accounting_oof.copy()
    )

    adjusted_predictions[
        :,
        ebitda_id
    ] = ebitda_predictions

    for _, val_idx in CV_SPLITS:

        ebitda_fold_scores.append(
            smape(
                y.iloc[val_idx]["Q0_EBITDA"],
                ebitda_predictions[val_idx]
            )
        )

        overall_fold_scores.append(
            evaluate_predictions(
                y.iloc[val_idx].to_numpy(),
                adjusted_predictions[val_idx]
            )["AVERAGE"]
        )

    da_comparison.append({
        "method": method,
        "da_oof_smape": smape(
            q0_da,
            da_predictions
        ),
        "ebitda_cv_smape": np.mean(
            ebitda_fold_scores
        ),
        "ebitda_cv_std": np.std(
            ebitda_fold_scores,
            ddof=1
        ),
        "overall_cv_smape": np.mean(
            overall_fold_scores
        ),
        "overall_cv_std": np.std(
            overall_fold_scores,
            ddof=1
        )
    })


da_comparison = (
    pd.DataFrame(da_comparison)
    .sort_values("ebitda_cv_smape")
    .reset_index(drop=True)
)

display(da_comparison)


# select the best D&A method
best_da_method = da_comparison.loc[
    0,
    "method"
]

xgb_final_oof = xgb_accounting_oof.copy()

xgb_final_oof[:, ebitda_id] = (
    da_ebitda_predictions[
        best_da_method
    ]
)


# calculate final results
xgb_final_scores = evaluate_predictions(
    y.to_numpy(),
    xgb_final_oof
)

xgb_final_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        xgb_final_scores[target]
        for target in TARGETS
    ] + [
        xgb_final_scores["AVERAGE"]
    ]
})


print(
    "Best D&A method:",
    best_da_method
)

print(
    f"Final overall sMAPE: "
    f"{xgb_final_scores['AVERAGE']:.4f}"
)

display(xgb_final_target_results)


Fold 1/5
  Ridge | D&A sMAPE=14.7887 | EBITDA sMAPE=39.2124 | time=0.5s
  XGBoost | D&A sMAPE=21.4888 | EBITDA sMAPE=39.3611 | best iteration=2998 | time=4.4 minutes

Fold 2/5
  Ridge | D&A sMAPE=15.6867 | EBITDA sMAPE=38.8225 | time=0.2s
  XGBoost | D&A sMAPE=22.2611 | EBITDA sMAPE=38.9918 | best iteration=2999 | time=4.6 minutes

Fold 3/5
  Ridge | D&A sMAPE=14.9484 | EBITDA sMAPE=39.3117 | time=0.2s
  XGBoost | D&A sMAPE=21.5976 | EBITDA sMAPE=39.4395 | best iteration=2999 | time=4.6 minutes

Fold 4/5
  Ridge | D&A sMAPE=14.9516 | EBITDA sMAPE=39.5325 | time=0.2s
  XGBoost | D&A sMAPE=21.8160 | EBITDA sMAPE=39.7549 | best iteration=2999 | time=4.6 minutes

Fold 5/5
  Ridge | D&A sMAPE=15.1328 | EBITDA sMAPE=39.1226 | time=0.2s
  XGBoost | D&A sMAPE=21.9428 | EBITDA sMAPE=39.3186 | best iteration=2998 | time=4.7 minutes


,method,da_oof_smape,ebitda_cv_smape,ebitda_cv_std,overall_cv_smape,overall_cv_std
0,Q1 Carry-Forward,2.3389,39.1992,0.2600,14.0320,0.1024
1,Ridge,15.1016,39.2003,0.2606,14.0321,0.1024
2,XGBoost,21.8213,39.3732,0.2733,14.0513,0.1041


Best D&A method: Q1 Carry-Forward
Final overall sMAPE: 14.0320


,target,smape
0,Q0_TOTAL_ASSETS,2.2363
1,Q0_TOTAL_LIABILITIES,3.7211
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,7.6258
3,Q0_GROSS_PROFIT,17.9928
4,Q0_COST_OF_REVENUES,7.5704
5,Q0_REVENUES,5.0352
6,Q0_OPERATING_INCOME,40.8649
7,Q0_OPERATING_EXPENSES,2.0420
8,Q0_EBITDA,39.1992
9,AVERAGE,14.0320


### Q0 D&A Prediction Results

Q1 carry-forward produced the best result, with a D&A sMAPE of 2.3389 and a derived EBITDA sMAPE of 39.1992. Ridge and XGBoost performed worse despite their additional complexity. This suggests that depreciation and amortization is highly stable between adjacent quarters, making the most recent value a stronger predictor than the fitted models. Therefore, Q1 D&A was retained, resulting in a final accounting-adjusted XGBoost sMAPE of 14.0320 ± 0.1024.

## Composite Model

The accounting-adjusted XGBoost and Random Forest predictions were combined to test whether their different prediction errors could offset each other. Several weighted averages were evaluated using the existing out-of-fold predictions, with XGBoost weights ranging from 0.5 to 1.0. The weight producing the lowest five-fold CV sMAPE was selected. If no blend outperformed XGBoost alone, the accounting-adjusted XGBoost model was retained.

In [18]:
# define ensemble weights
xgb_weights = np.round(
    np.arange(
        0.50,
        1.001,
        0.01
    ),
    2
)
ensemble_results = []
ensemble_predictions = {}


# evaluate each weight
for xgb_weight in xgb_weights:

    xgb_weight = round(
        float(xgb_weight),
        2
    )

    rf_weight = round(
        1 - xgb_weight,
        2
    )

    blended_oof = (
        xgb_weight * xgb_accounting_oof +
        rf_weight * rf_accounting_oof
    )

    ensemble_predictions[
        xgb_weight
    ] = blended_oof

    fold_scores = []

    for _, val_idx in CV_SPLITS:

        fold_score = evaluate_predictions(
            y.iloc[val_idx].to_numpy(),
            blended_oof[val_idx]
        )["AVERAGE"]

        fold_scores.append(fold_score)

    ensemble_results.append({
        "xgb_weight": xgb_weight,
        "rf_weight": rf_weight,
        "mean_cv_smape": np.mean(
            fold_scores
        ),
        "std_cv_smape": np.std(
            fold_scores,
            ddof=1
        )
    })


# summarize ensemble weights
ensemble_results = (
    pd.DataFrame(ensemble_results)
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(ensemble_results)


# select the best ensemble
best_xgb_weight = ensemble_results.loc[
    0,
    "xgb_weight"
]

best_rf_weight = ensemble_results.loc[
    0,
    "rf_weight"
]

best_ensemble_oof = ensemble_predictions[
    best_xgb_weight
]


# calculate final ensemble results
ensemble_scores = evaluate_predictions(
    y.to_numpy(),
    best_ensemble_oof
)

ensemble_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        ensemble_scores[target]
        for target in TARGETS
    ] + [
        ensemble_scores["AVERAGE"]
    ]
})


print(
    f"Best weights: "
    f"XGBoost={best_xgb_weight:.2f}, "
    f"Random Forest={best_rf_weight:.2f}"
)

print(
    f"Best ensemble CV sMAPE: "
    f"{ensemble_results.loc[0, 'mean_cv_smape']:.4f} ± "
    f"{ensemble_results.loc[0, 'std_cv_smape']:.4f}"
)

if best_xgb_weight == 1.0:
    print(
        "The ensemble did not improve upon "
        "accounting-adjusted XGBoost."
    )
else:
    print(
        "The ensemble improved upon "
        "accounting-adjusted XGBoost."
    )

display(ensemble_target_results)

,xgb_weight,rf_weight,mean_cv_smape,std_cv_smape
0,0.9300,0.0700,13.9664,0.1020
1,0.9200,0.0800,13.9677,0.1008
2,0.9400,0.0600,13.9680,0.1025
3,0.9500,0.0500,13.9717,0.1024
4,0.9100,0.0900,13.9722,0.1005
5,0.9600,0.0400,13.9783,0.1023
6,0.9000,0.1000,13.9795,0.1002
7,0.9700,0.0300,13.9875,0.1021
8,0.8900,0.1100,13.9891,0.0999
9,0.9800,0.0200,13.9994,0.1014


Best weights: XGBoost=0.93, Random Forest=0.07
Best ensemble CV sMAPE: 13.9664 ± 0.1020
The ensemble improved upon accounting-adjusted XGBoost.


,target,smape
0,Q0_TOTAL_ASSETS,2.2663
1,Q0_TOTAL_LIABILITIES,3.8077
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,7.4904
3,Q0_GROSS_PROFIT,17.7427
4,Q0_COST_OF_REVENUES,7.5880
5,Q0_REVENUES,5.0764
6,Q0_OPERATING_INCOME,40.6333
7,Q0_OPERATING_EXPENSES,2.1268
8,Q0_EBITDA,38.9659
9,AVERAGE,13.9664


# Conclusion - Non-Deep-Learning Model

The non-deep-learning models were evaluated using the same five cross-validation folds and original-scale sMAPE. Elastic Net performed poorly, suggesting that the relationships between the predictors and financial targets are highly nonlinear. Random Forest achieved a CV sMAPE of 27.3906, while XGBoost improved this to 23.2273.

Accounting-based post-processing produced the largest improvement. Gross profit was derived from predicted revenue and cost of revenues, operating income was derived from gross profit and operating expenses, and EBITDA was derived using operating income and Q1 depreciation and amortization. This reduced the XGBoost CV sMAPE to 14.0320. Using Q1 D&A directly also outperformed Ridge and XGBoost models designed to predict Q0 D&A, indicating strong quarter-to-quarter stability.

Finally, the accounting-adjusted XGBoost and Random Forest predictions were blended. The best ensemble assigned 93% weight to XGBoost and 7% to Random Forest, producing a CV sMAPE of 13.9664 ± 0.1020. The small Random Forest weight provided a modest improvement while confirming that XGBoost remained the dominant model. Operating income and EBITDA remained the most difficult targets, with sMAPEs of 40.6333 and 38.9659. The 93/7 ensemble was therefore selected as the final non-deep-learning approach.

### Final Non-Deep-Learning Model Training and Saving

After selecting the best non-deep-learning approach through five-fold cross-validation, we retrained its XGBoost and Random Forest components using the complete training dataset. The accounting relationships were then applied to derive gross profit, operating income, and EBITDA. Test predictions from the two models were combined using the selected weights of 93% XGBoost and 7% Random Forest. Finally, both the fitted models and test predictions were saved so they can be reused without repeating the time-consuming training process.


In [19]:
import joblib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder


# create output folder
SAVE_DIRECTORY = ARTIFACT_DIR / "saved_non_dl_model"
SAVE_DIRECTORY.mkdir(exist_ok=True)


# define final settings
XGB_WEIGHT = 0.93
RF_WEIGHT = 0.07

XGB_N_ESTIMATORS = 2321

DIRECT_TARGETS = [
    "Q0_TOTAL_ASSETS",
    "Q0_TOTAL_LIABILITIES",
    "Q0_TOTAL_STOCKHOLDERS_EQUITY",
    "Q0_COST_OF_REVENUES",
    "Q0_REVENUES",
    "Q0_OPERATING_EXPENSES"
]


# preprocess predictors for XGBoost
xgb_final_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=True,
                        dtype=np.float32
                    )
                )
            ]),
            categorical_columns
        )
    ],
    sparse_threshold=1.0
)


# fit XGBoost preprocessing
print("Preprocessing data for XGBoost...")

X_xgb_full = xgb_final_preprocessor.fit_transform(X)
X_xgb_test = xgb_final_preprocessor.transform(X_test)


# train final XGBoost models
xgb_final_models = {}
xgb_test_predictions = np.zeros(
    (len(X_test), len(TARGETS))
)

xgb_params = {
    "objective": "reg:squarederror",
    "n_estimators": XGB_N_ESTIMATORS,
    "learning_rate": 0.03,
    "max_depth": 8,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "max_bin": 256,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": 0
}

print("\nTraining final XGBoost models...")

for target in DIRECT_TARGETS:

    start_time = time.time()

    model = xgb.XGBRegressor(**xgb_params)

    model.fit(
        X_xgb_full,
        y_transformed[target]
    )

    prediction_transformed = model.predict(
        X_xgb_test
    )

    target_id = TARGETS.index(target)

    xgb_test_predictions[:, target_id] = (
        inverse_signed_log1p(
            prediction_transformed
        )
    )

    xgb_final_models[target] = model

    elapsed = time.time() - start_time

    print(
        f"{target} completed | "
        f"time={elapsed / 60:.1f} minutes"
    )


# derive XGBoost accounting targets
revenue_id = TARGETS.index("Q0_REVENUES")
cost_id = TARGETS.index("Q0_COST_OF_REVENUES")
gross_profit_id = TARGETS.index("Q0_GROSS_PROFIT")
operating_expenses_id = TARGETS.index(
    "Q0_OPERATING_EXPENSES"
)
operating_income_id = TARGETS.index(
    "Q0_OPERATING_INCOME"
)
ebitda_id = TARGETS.index("Q0_EBITDA")

xgb_test_predictions[:, gross_profit_id] = (
    xgb_test_predictions[:, revenue_id]
    - xgb_test_predictions[:, cost_id]
)

xgb_test_predictions[:, operating_income_id] = (
    xgb_test_predictions[:, gross_profit_id]
    - xgb_test_predictions[:, operating_expenses_id]
)

xgb_test_predictions[:, ebitda_id] = (
    xgb_test_predictions[:, operating_income_id]
    + X_test[
        "Q1_DEPRECIATION_AND_AMORTIZATION"
    ].to_numpy()
)


# save XGBoost checkpoint
xgb_checkpoint = {
    "preprocessor": xgb_final_preprocessor,
    "models": xgb_final_models,
    "params": xgb_params,
    "direct_targets": DIRECT_TARGETS,
    "targets": TARGETS
}

joblib.dump(
    xgb_checkpoint,
    SAVE_DIRECTORY / "xgboost_checkpoint.joblib"
)

np.save(
    SAVE_DIRECTORY / "xgboost_test_predictions.npy",
    xgb_test_predictions
)

print("\nXGBoost checkpoint saved.")


# preprocess predictors for Random Forest
rf_final_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "ordinal",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    )
                )
            ]),
            categorical_columns
        )
    ]
)


# fit Random Forest preprocessing
print("\nPreprocessing data for Random Forest...")

X_rf_full = rf_final_preprocessor.fit_transform(X)
X_rf_test = rf_final_preprocessor.transform(X_test)


# train final Random Forest
print("\nTraining final Random Forest...")
rf_start_time = time.time()

rf_final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=25,
    min_samples_leaf=2,
    max_features=0.5,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

rf_final_model.fit(
    X_rf_full,
    y_transformed
)

rf_test_predictions_transformed = (
    rf_final_model.predict(X_rf_test)
)

rf_test_predictions = inverse_signed_log1p(
    rf_test_predictions_transformed
)

print(
    f"Random Forest completed | "
    f"time={(time.time() - rf_start_time) / 60:.1f} minutes"
)


# derive Random Forest accounting targets
rf_test_predictions[:, gross_profit_id] = (
    rf_test_predictions[:, revenue_id]
    - rf_test_predictions[:, cost_id]
)

rf_test_predictions[:, operating_income_id] = (
    rf_test_predictions[:, gross_profit_id]
    - rf_test_predictions[:, operating_expenses_id]
)

rf_test_predictions[:, ebitda_id] = (
    rf_test_predictions[:, operating_income_id]
    + X_test[
        "Q1_DEPRECIATION_AND_AMORTIZATION"
    ].to_numpy()
)


# blend the final predictions
final_test_predictions = (
    XGB_WEIGHT * xgb_test_predictions
    + RF_WEIGHT * rf_test_predictions
)


# create the final prediction file
final_predictions = pd.DataFrame(
    final_test_predictions,
    columns=TARGETS
)

final_predictions.insert(
    0,
    ID_COLUMN,
    test_ids.to_numpy()
)

final_predictions.to_csv(
    SAVE_DIRECTORY / "final_non_dl_predictions.csv",
    index=False
)


# save the complete final model
final_model_bundle = {
    "targets": TARGETS,
    "id_column": ID_COLUMN,
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
    "xgb_preprocessor": xgb_final_preprocessor,
    "xgb_models": xgb_final_models,
    "xgb_params": xgb_params,
    "xgb_direct_targets": DIRECT_TARGETS,
    "rf_preprocessor": rf_final_preprocessor,
    "rf_model": rf_final_model,
    "xgb_weight": XGB_WEIGHT,
    "rf_weight": RF_WEIGHT,
    "da_column": "Q1_DEPRECIATION_AND_AMORTIZATION"
}

joblib.dump(
    final_model_bundle,
    SAVE_DIRECTORY / "final_non_dl_model.joblib",
    compress=3
)


# verify saved files
print("\nSaved files:")

for file_path in SAVE_DIRECTORY.iterdir():
    print(
        f"{file_path.name}: "
        f"{file_path.stat().st_size / 1_000_000:.1f} MB"
    )

print("\nFinal prediction shape:", final_predictions.shape)
display(final_predictions.head())

Preprocessing data for XGBoost...

Training final XGBoost models...
Q0_TOTAL_ASSETS completed | time=1.5 minutes
Q0_TOTAL_LIABILITIES completed | time=1.6 minutes
Q0_TOTAL_STOCKHOLDERS_EQUITY completed | time=1.7 minutes
Q0_COST_OF_REVENUES completed | time=1.9 minutes
Q0_REVENUES completed | time=2.5 minutes
Q0_OPERATING_EXPENSES completed | time=2.6 minutes

XGBoost checkpoint saved.

Preprocessing data for Random Forest...

Training final Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  9.0min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 10.0min finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.5s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.6s finished


Random Forest completed | time=10.0 minutes

Saved files:
final_non_dl_model.joblib: 473.7 MB
final_non_dl_predictions.csv: 8.7 MB
xgboost_checkpoint.joblib: 86.8 MB
xgboost_test_predictions.npy: 3.6 MB

Final prediction shape: (50000, 10)


,Id,Q0_TOTAL_ASSETS,Q0_TOTAL_LIABILITIES,Q0_TOTAL_STOCKHOLDERS_EQUITY,Q0_GROSS_PROFIT,Q0_COST_OF_REVENUES,Q0_REVENUES,Q0_OPERATING_INCOME,Q0_OPERATING_EXPENSES,Q0_EBITDA
0,171381,"166,544,380.3992","66,979,227.7123","100,736,084.6948","7,284,385.7563","8,610,839.9339","15,895,225.6902","-6,383,310.7405","13,667,696.4968","-6,546,685.7763"
1,171809,"61,489,024.8657","20,369,824.5549","41,212,802.0329","2,039,865.5384","7,642,669.4533","9,682,534.9917","-7,007,033.4643","9,046,899.0027","-6,951,663.9521"
2,109665,"16,279,181.5377","11,683,733.8690","1,510,080.0992","76,813.1870","1,069,847.0193","1,146,660.2063","-1,729,422.4779","1,806,235.6648","-1,740,543.9807"
3,122380,"76,557,540.5905","40,466,938.2824","27,463,944.5999","519,981.1533","1,279,438.8701","1,799,420.0234","-6,297,626.2310","6,817,607.3843","-5,255,865.6792"
4,140286,"223,381,149.1579","68,672,414.5574","158,575,233.9188","4,717,600.9964","9,620,989.4773","14,338,590.4738","-19,627,043.8163","24,344,644.8128","-19,570,554.6695"


# Deep Learning Models

### Proprocess

In [5]:
import random
import numpy as np
import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# set random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# select training device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Seed:", SEED)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# identify predictor types
dl_numeric_columns = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

dl_categorical_columns = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)


# check missing values
nan_count = int(X.isna().sum().sum())

missing_token_counts = (
    X[dl_categorical_columns]
    .apply(
        lambda column: (
            column.astype(str)
            .str.strip()
            .str.upper()
            .eq("MISSING")
            .sum()
        )
    )
)

print("Numerical predictors:", len(dl_numeric_columns))
print("Categorical predictors:", len(dl_categorical_columns))
print("NaN values:", nan_count)
print(
    'Values labelled "MISSING":',
    int(missing_token_counts.sum())
)

display(
    missing_token_counts[
        missing_token_counts > 0
    ].rename("missing_count")
)


# confirm numerical predictors have no missing values
assert (
    X[dl_numeric_columns]
    .isna()
    .sum()
    .sum()
    == 0
)


# create preprocessing pipeline
def make_dl_preprocessor():

    # standardize numerical predictors
    numeric_pipeline = Pipeline([
        (
            "scaler",
            StandardScaler()
        )
    ])

    # encode categorical predictors
    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.float32
            )
        )
    ])

    return ColumnTransformer(
        [
            (
                "numeric",
                numeric_pipeline,
                dl_numeric_columns
            ),
            (
                "categorical",
                categorical_pipeline,
                dl_categorical_columns
            )
        ],
        remainder="drop"
    )


# define an unfitted preprocessing pipeline
dl_preprocessor = make_dl_preprocessor()

print("Deep-learning preprocessing pipeline created.")

Seed: 937
Device: cpu
Numerical predictors: 198
Categorical predictors: 4
NaN values: 0
Values labelled "MISSING": 3808


industry             1344
financialCurrency    1232
recommendationKey    1232
Name: missing_count, dtype: int64

Deep-learning preprocessing pipeline created.


Before training the neural networks, numerical and categorical predictors were processed separately. Missing numerical values were replaced with the median calculated from the training fold, and missing-value indicators were added so the model could learn whether missingness contained useful information. Numerical predictors were then standardized. Missing categorical values were treated as a separate category before one-hot encoding. All preprocessing steps were fitted only on each training fold and then applied to its validation fold, preventing data leakage. The global seed of 937 was also applied throughout to make model initialization, dropout, and training results reproducible.

We considered six feed-forward neural-network configurations. The wide MLP uses one large hidden layer to test whether greater breadth can capture the relationships in the data. The deep MLP uses four progressively smaller hidden layers to learn more hierarchical representations. Three versions of the deep MLP compare dropout rates of 0%, 10%, and 20%, while another version removes early stopping to evaluate its effect on overfitting. Finally, the residual MLP uses skip connections that add each block’s input to its output, which can improve gradient flow and make deeper networks easier to train. All models initially use ReLU activation, batch normalization, He initialization, and the same training settings to ensure a fair comparison.

In short, the six configurations are:

- Wide MLP with 10% dropout and early stopping
- Deep MLP without dropout and with early stopping
- Deep MLP with 10% dropout and early stopping
- Deep MLP with 20% dropout and early stopping
- Deep MLP with 10% dropout and no early stopping
- Residual MLP with 10% dropout and early stopping

Early stopping use patience of 10.

In [6]:
import torch.nn as nn
import pandas as pd


# create activation function
def make_activation(name):

    if name == "relu":
        return nn.ReLU()

    if name == "leaky_relu":
        return nn.LeakyReLU(negative_slope=0.01)

    raise ValueError(f"Unknown activation: {name}")


# define standard MLP
class MLP(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers,
        dropout,
        activation
    ):

        super().__init__()

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_layers:

            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                make_activation(activation)
            ])

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            current_dim = hidden_dim

        layers.append(
            nn.Linear(current_dim, output_dim)
        )

        self.network = nn.Sequential(*layers)

        self.initialize_weights(activation)

    def initialize_weights(self, activation):

        negative_slope = (
            0.01 if activation == "leaky_relu"
            else 0
        )

        for layer in self.modules():

            if isinstance(layer, nn.Linear):

                nn.init.kaiming_normal_(
                    layer.weight,
                    a=negative_slope,
                    nonlinearity=(
                        "leaky_relu"
                        if activation == "leaky_relu"
                        else "relu"
                    )
                )

                nn.init.zeros_(layer.bias)

    def forward(self, x):

        return self.network(x)


# define residual block
class ResidualBlock(nn.Module):

    def __init__(
        self,
        width,
        dropout,
        activation
    ):

        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(width, width),
            nn.BatchNorm1d(width),
            make_activation(activation),
            nn.Dropout(dropout),
            nn.Linear(width, width),
            nn.BatchNorm1d(width)
        )

        self.activation = make_activation(activation)

    def forward(self, x):

        return self.activation(
            x + self.block(x)
        )


# define residual MLP
class ResidualMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
        width,
        number_of_blocks,
        dropout,
        activation
    ):

        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            make_activation(activation)
        )

        self.residual_blocks = nn.Sequential(
            *[
                ResidualBlock(
                    width=width,
                    dropout=dropout,
                    activation=activation
                )
                for _ in range(number_of_blocks)
            ]
        )

        self.output_layer = nn.Linear(
            width,
            output_dim
        )

        self.initialize_weights(activation)

    def initialize_weights(self, activation):

        negative_slope = (
            0.01 if activation == "leaky_relu"
            else 0
        )

        for layer in self.modules():

            if isinstance(layer, nn.Linear):

                nn.init.kaiming_normal_(
                    layer.weight,
                    a=negative_slope,
                    nonlinearity=(
                        "leaky_relu"
                        if activation == "leaky_relu"
                        else "relu"
                    )
                )

                nn.init.zeros_(layer.bias)

    def forward(self, x):

        x = self.input_layer(x)
        x = self.residual_blocks(x)

        return self.output_layer(x)


# define architecture configurations
dl_configs = [
    {
        "name": "Wide MLP",
        "model_type": "mlp",
        "hidden_layers": [1024],
        "dropout": 0.10,
        "activation": "relu",
        "early_stopping": True
    },
    {
        "name": "Deep MLP - No Dropout",
        "model_type": "mlp",
        "hidden_layers": [512, 256, 128, 64],
        "dropout": 0.00,
        "activation": "relu",
        "early_stopping": True
    },
    {
        "name": "Deep MLP - Dropout 0.10",
        "model_type": "mlp",
        "hidden_layers": [512, 256, 128, 64],
        "dropout": 0.10,
        "activation": "relu",
        "early_stopping": True
    },
    {
        "name": "Deep MLP - Dropout 0.20",
        "model_type": "mlp",
        "hidden_layers": [512, 256, 128, 64],
        "dropout": 0.20,
        "activation": "relu",
        "early_stopping": True
    },
    {
        "name": "Deep MLP - No Early Stopping",
        "model_type": "mlp",
        "hidden_layers": [512, 256, 128, 64],
        "dropout": 0.10,
        "activation": "relu",
        "early_stopping": False
    },
    {
        "name": "Residual MLP",
        "model_type": "residual",
        "width": 256,
        "number_of_blocks": 3,
        "dropout": 0.10,
        "activation": "relu",
        "early_stopping": True
    }
]


# define common training settings
DL_BATCH_SIZE = 512
DL_MAX_EPOCHS = 200
DL_PATIENCE = 10
DL_LEARNING_RATE = 0.001
DL_WEIGHT_DECAY = 0.0


# display configurations
dl_config_table = pd.DataFrame([
    {
        "model": config["name"],
        "architecture": (
            config["hidden_layers"]
            if config["model_type"] == "mlp"
            else (
                f"{config['number_of_blocks']} "
                f"residual blocks × {config['width']}"
            )
        ),
        "dropout": config["dropout"],
        "activation": config["activation"],
        "early_stopping": config["early_stopping"]
    }
    for config in dl_configs
])

display(dl_config_table)

,model,architecture,dropout,activation,early_stopping
0,Wide MLP,[1024],0.1000,relu,True
1,Deep MLP - No Dropout,"[512, 256, 128, 64]",0.0000,relu,True
2,Deep MLP - Dropout 0.10,"[512, 256, 128, 64]",0.1000,relu,True
3,Deep MLP - Dropout 0.20,"[512, 256, 128, 64]",0.2000,relu,True
4,Deep MLP - No Early Stopping,"[512, 256, 128, 64]",0.1000,relu,False
5,Residual MLP,3 residual blocks × 256,0.1000,relu,True


In [7]:
import copy
import time

from torch.utils.data import DataLoader, TensorDataset


# apply accounting relationships
def apply_dl_accounting(predictions, predictors):

    predictions = predictions.copy()

    revenue_id = TARGETS.index("Q0_REVENUES")
    cost_id = TARGETS.index("Q0_COST_OF_REVENUES")
    gross_profit_id = TARGETS.index("Q0_GROSS_PROFIT")
    operating_expenses_id = TARGETS.index(
        "Q0_OPERATING_EXPENSES"
    )
    operating_income_id = TARGETS.index(
        "Q0_OPERATING_INCOME"
    )
    ebitda_id = TARGETS.index("Q0_EBITDA")

    predictions[:, gross_profit_id] = (
        predictions[:, revenue_id]
        - predictions[:, cost_id]
    )

    predictions[:, operating_income_id] = (
        predictions[:, gross_profit_id]
        - predictions[:, operating_expenses_id]
    )

    predictions[:, ebitda_id] = (
        predictions[:, operating_income_id]
        + predictors[
            "Q1_DEPRECIATION_AND_AMORTIZATION"
        ].to_numpy()
    )

    return predictions


# create a model
def build_dl_model(config, input_dim):

    if config["model_type"] == "mlp":

        return MLP(
            input_dim=input_dim,
            output_dim=len(TARGETS),
            hidden_layers=config["hidden_layers"],
            dropout=config["dropout"],
            activation=config["activation"]
        )

    return ResidualMLP(
        input_dim=input_dim,
        output_dim=len(TARGETS),
        width=config["width"],
        number_of_blocks=config["number_of_blocks"],
        dropout=config["dropout"],
        activation=config["activation"]
    )


# generate predictions in batches
def predict_dl(model, predictors, batch_size=4096):

    model.eval()

    dataset = TensorDataset(
        torch.from_numpy(predictors)
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    predictions = []

    with torch.no_grad():

        for (batch_x,) in loader:

            batch_x = batch_x.to(DEVICE)

            batch_predictions = model(
                batch_x
            ).cpu().numpy()

            predictions.append(batch_predictions)

    return np.vstack(predictions)


# initialize results
dl_config_results = {
    config_id: {
        "config": config,
        "oof_predictions": np.zeros(
            (len(y), len(TARGETS))
        ),
        "fold_scores": [],
        "epochs": []
    }
    for config_id, config in enumerate(
        dl_configs,
        start=1
    )
}


# train with five-fold cross-validation
for fold, (train_idx, val_idx) in enumerate(
    CV_SPLITS,
    start=1
):

    print(f"\n{'=' * 70}")
    print(f"Fold {fold}/{N_FOLDS}")
    print(f"{'=' * 70}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_original = y.iloc[train_idx]
    y_val_original = y.iloc[val_idx]

    y_train_log = (
        y_transformed.iloc[train_idx]
        .to_numpy(dtype=np.float32)
    )

    y_val_log = (
        y_transformed.iloc[val_idx]
        .to_numpy(dtype=np.float32)
    )

    # fit preprocessing on the training fold
    fold_preprocessor = make_dl_preprocessor()

    X_train_processed = (
        fold_preprocessor
        .fit_transform(X_train)
        .astype(np.float32)
    )

    X_val_processed = (
        fold_preprocessor
        .transform(X_val)
        .astype(np.float32)
    )

    # standardize transformed targets
    target_mean = y_train_log.mean(
        axis=0,
        keepdims=True
    )

    target_std = y_train_log.std(
        axis=0,
        keepdims=True
    )

    target_std[target_std == 0] = 1.0

    y_train_scaled = (
        (y_train_log - target_mean)
        / target_std
    ).astype(np.float32)

    y_val_scaled = (
        (y_val_log - target_mean)
        / target_std
    ).astype(np.float32)

    assert np.isfinite(
        X_train_processed
    ).all()

    assert np.isfinite(
        X_val_processed
    ).all()

    input_dim = X_train_processed.shape[1]

    print("Processed predictors:", input_dim)

    # train every configuration
    for config_id, result in dl_config_results.items():

        config = result["config"]

        print(f"\nConfig {config_id}/{len(dl_configs)}")
        print(config["name"])

        # reset random seeds
        model_seed = SEED + fold

        random.seed(model_seed)
        np.random.seed(model_seed)
        torch.manual_seed(model_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(model_seed)

        # create training loader
        train_generator = torch.Generator()
        train_generator.manual_seed(model_seed)

        train_dataset = TensorDataset(
            torch.from_numpy(X_train_processed),
            torch.from_numpy(y_train_scaled)
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=DL_BATCH_SIZE,
            shuffle=True,
            generator=train_generator,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        # create model
        model = build_dl_model(
            config,
            input_dim
        ).to(DEVICE)

        criterion = nn.MSELoss()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=DL_LEARNING_RATE,
            weight_decay=DL_WEIGHT_DECAY
        )

        best_val_smape = np.inf
        best_state = None
        best_epoch = 0
        patience_counter = 0

        training_start = time.time()

        # train model
        for epoch in range(
            1,
            DL_MAX_EPOCHS + 1
        ):

            model.train()

            total_train_loss = 0.0

            for batch_x, batch_y in train_loader:

                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)

                optimizer.zero_grad()

                batch_predictions = model(batch_x)

                loss = criterion(
                    batch_predictions,
                    batch_y
                )

                loss.backward()
                optimizer.step()

                total_train_loss += (
                    loss.item()
                    * len(batch_x)
                )

            train_loss = (
                total_train_loss
                / len(train_dataset)
            )

            # calculate validation predictions
            val_predictions_scaled = predict_dl(
                model,
                X_val_processed
            )

            val_loss = np.mean(
                (
                    val_predictions_scaled
                    - y_val_scaled
                ) ** 2
            )

            # return predictions to original units
            val_predictions_log = (
                val_predictions_scaled
                * target_std
                + target_mean
            )

            val_predictions = (
                inverse_signed_log1p(
                    val_predictions_log
                )
            )

            val_predictions = apply_dl_accounting(
                val_predictions,
                X_val
            )

            val_smape = evaluate_predictions(
                y_val_original.to_numpy(),
                val_predictions
            )["AVERAGE"]

            # save best epoch
            if val_smape < best_val_smape - 1e-4:

                best_val_smape = val_smape
                best_epoch = epoch
                patience_counter = 0

                if config["early_stopping"]:
                    best_state = copy.deepcopy(
                        model.state_dict()
                    )

            else:
                patience_counter += 1

            # print training progress
            if (
                epoch == 1
                or epoch % 10 == 0
                or epoch == DL_MAX_EPOCHS
            ):

                train_predictions_scaled = predict_dl(
                    model,
                    X_train_processed
                )

                train_predictions_log = (
                    train_predictions_scaled
                    * target_std
                    + target_mean
                )

                train_predictions = (
                    inverse_signed_log1p(
                        train_predictions_log
                    )
                )

                train_predictions = apply_dl_accounting(
                    train_predictions,
                    X_train
                )

                train_smape = evaluate_predictions(
                    y_train_original.to_numpy(),
                    train_predictions
                )["AVERAGE"]

                print(
                    f"Epoch {epoch:03d} | "
                    f"train loss={train_loss:.4f} | "
                    f"val loss={val_loss:.4f} | "
                    f"train sMAPE={train_smape:.4f} | "
                    f"val sMAPE={val_smape:.4f}"
                )

            # apply early stopping
            if (
                config["early_stopping"]
                and patience_counter >= DL_PATIENCE
            ):

                print(
                    f"Early stopping at epoch {epoch} | "
                    f"best epoch={best_epoch}"
                )

                break

        epochs_trained = epoch

        # restore best model
        if (
            config["early_stopping"]
            and best_state is not None
        ):

            model.load_state_dict(best_state)

            selected_epoch = best_epoch

        else:
            selected_epoch = epochs_trained

        # calculate final fold predictions
        fold_predictions_scaled = predict_dl(
            model,
            X_val_processed
        )

        fold_predictions_log = (
            fold_predictions_scaled
            * target_std
            + target_mean
        )

        fold_predictions = inverse_signed_log1p(
            fold_predictions_log
        )

        fold_predictions = apply_dl_accounting(
            fold_predictions,
            X_val
        )

        # store out-of-fold predictions
        result["oof_predictions"][
            val_idx
        ] = fold_predictions

        # calculate fold score
        fold_score = evaluate_predictions(
            y_val_original.to_numpy(),
            fold_predictions
        )["AVERAGE"]

        result["fold_scores"].append(
            fold_score
        )

        result["epochs"].append(
            selected_epoch
        )

        elapsed = time.time() - training_start

        print(
            f"Fold sMAPE={fold_score:.4f} | "
            f"selected epoch={selected_epoch} | "
            f"time={elapsed / 60:.1f} minutes"
        )

        del model
        del optimizer

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# summarize configurations
dl_results = []

for config_id, result in dl_config_results.items():

    dl_results.append({
        "config_id": config_id,
        "model": result["config"]["name"],
        "mean_cv_smape": np.mean(
            result["fold_scores"]
        ),
        "std_cv_smape": np.std(
            result["fold_scores"],
            ddof=1
        ),
        "mean_selected_epoch": np.mean(
            result["epochs"]
        )
    })

dl_results = (
    pd.DataFrame(dl_results)
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(dl_results)


# select best configuration
best_dl_config_id = int(
    dl_results.loc[0, "config_id"]
)

best_dl_result = dl_config_results[
    best_dl_config_id
]

best_dl_config = best_dl_result["config"]
dl_oof = best_dl_result["oof_predictions"]

dl_scores = evaluate_predictions(
    y.to_numpy(),
    dl_oof
)

print("Best DL model:", best_dl_config["name"])

print(
    f"Best DL CV sMAPE: "
    f"{dl_results.loc[0, 'mean_cv_smape']:.4f} ± "
    f"{dl_results.loc[0, 'std_cv_smape']:.4f}"
)


# display target-level results
dl_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        dl_scores[target]
        for target in TARGETS
    ] + [dl_scores["AVERAGE"]]
})

display(dl_target_results)


Fold 1/5
Processed predictors: 332

Config 1/6
Wide MLP
Epoch 001 | train loss=0.4851 | val loss=0.2763 | train sMAPE=69.9183 | val sMAPE=70.3498
Epoch 010 | train loss=0.2021 | val loss=0.1841 | train sMAPE=50.2379 | val sMAPE=50.6141
Epoch 020 | train loss=0.1791 | val loss=0.1674 | train sMAPE=43.2321 | val sMAPE=43.9146
Epoch 030 | train loss=0.1687 | val loss=0.1599 | train sMAPE=43.7859 | val sMAPE=44.3534
Epoch 040 | train loss=0.1619 | val loss=0.1545 | train sMAPE=45.6502 | val sMAPE=46.1427
Epoch 050 | train loss=0.1563 | val loss=0.1525 | train sMAPE=46.5759 | val sMAPE=47.0657
Epoch 060 | train loss=0.1514 | val loss=0.1485 | train sMAPE=44.8137 | val sMAPE=45.3691
Epoch 070 | train loss=0.1480 | val loss=0.1456 | train sMAPE=42.3723 | val sMAPE=42.8942
Epoch 080 | train loss=0.1470 | val loss=0.1455 | train sMAPE=44.1377 | val sMAPE=44.7676
Early stopping at epoch 86 | best epoch=76
Fold sMAPE=37.8597 | selected epoch=76 | time=6.2 minutes

Config 2/6
Deep MLP - No Dropou

,config_id,model,mean_cv_smape,std_cv_smape,mean_selected_epoch
0,5,Deep MLP - No Early Stopping,32.1771,3.4982,200.0000
1,3,Deep MLP - Dropout 0.10,32.4675,1.6545,40.8000
2,4,Deep MLP - Dropout 0.20,34.6276,1.3175,47.4000
3,6,Residual MLP,36.0721,1.3228,25.4000
4,2,Deep MLP - No Dropout,36.3801,1.5431,31.0000
5,1,Wide MLP,40.4071,2.5303,50.6000


Best DL model: Deep MLP - No Early Stopping
Best DL CV sMAPE: 32.1771 ± 3.4982


,target,smape
0,Q0_TOTAL_ASSETS,24.6773
1,Q0_TOTAL_LIABILITIES,28.7661
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,30.3898
3,Q0_GROSS_PROFIT,33.3882
4,Q0_COST_OF_REVENUES,30.1423
5,Q0_REVENUES,21.0464
6,Q0_OPERATING_INCOME,53.7813
7,Q0_OPERATING_EXPENSES,15.6989
8,Q0_EBITDA,51.7033
9,AVERAGE,32.1771


Among the initial architectures, the deep MLP with 10% dropout provided the strongest balance between bias and variance. It outperformed the wide MLP, the deep model without dropout, the model with 20% dropout, and the residual MLP while maintaining relatively stable performance across folds. The model without early stopping achieved a slightly lower mean cross-validation sMAPE, but its results varied considerably across folds because the final epoch was not consistently the best one. The training history also showed that patience of 10 sometimes stopped the model before later improvements occurred. Therefore, the deep MLP with 10% dropout appears to be the most promising architecture, but its training procedure should retain the best checkpoint over a longer training period instead of relying on either early stopping or the final epoch alone.

Before selecting the final deep-learning model, we will test two additional ideas. First, we will evaluate wider and deeper versions of the current MLP to determine whether additional capacity improves its ability to learn the complex relationships among the predictors. Second, we will test a hybrid GRU model that processes the Q10-to-Q1 financial histories as ordered sequences while a separate MLP processes the remaining static and categorical predictors. The two learned representations will then be combined to generate the nine predictions, followed by the same accounting adjustments. This comparison will show whether explicitly modeling the quarterly order provides information that a standard feed-forward network may overlook.

```text
160 quarterly predictors                   42 static predictors
          │                                         │
          ▼                                         ▼
 Reshape to 10 × 16                       Scale / one-hot encode
          │                                         │
          ▼                                         ▼
      GRU branch                              Static MLP branch
          │                                         │
          ▼                                         ▼
64-unit history summary                   128-unit static summary
          │                                         │
          └──────────────────┬──────────────────────┘
                             ▼
                  Concatenate both summaries
                             │
                             ▼
                       Combined MLP
                             │
                             ▼
                  Nine target predictions
                             │
                             ▼
                   Accounting adjustments
```

### Wider / Deeper Neural Network

In [8]:
# define expanded MLP configurations
expanded_mlp_configs = [
    {
        "name": "Reference Deep MLP",
        "model_type": "mlp",
        "hidden_layers": [512, 256, 128, 64],
        "dropout": 0.10,
        "activation": "relu"
    },
    {
        "name": "Wider Deep MLP",
        "model_type": "mlp",
        "hidden_layers": [1024, 512, 256, 128],
        "dropout": 0.10,
        "activation": "relu"
    },
    {
        "name": "Deeper MLP",
        "model_type": "mlp",
        "hidden_layers": [
            512, 512, 256, 256, 128, 64
        ],
        "dropout": 0.10,
        "activation": "relu"
    }
]


# initialize results
expanded_mlp_config_results = {
    config_id: {
        "config": config,
        "oof_predictions": np.zeros(
            (len(y), len(TARGETS))
        ),
        "fold_scores": [],
        "best_epochs": []
    }
    for config_id, config in enumerate(
        expanded_mlp_configs,
        start=1
    )
}


# train with five-fold cross-validation
for fold, (train_idx, val_idx) in enumerate(
    CV_SPLITS,
    start=1
):

    print(f"\n{'=' * 70}")
    print(f"Fold {fold}/{N_FOLDS}")
    print(f"{'=' * 70}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_original = y.iloc[train_idx]
    y_val_original = y.iloc[val_idx]

    y_train_log = (
        y_transformed.iloc[train_idx]
        .to_numpy(dtype=np.float32)
    )

    y_val_log = (
        y_transformed.iloc[val_idx]
        .to_numpy(dtype=np.float32)
    )

    # fit fold preprocessing
    fold_preprocessor = make_dl_preprocessor()

    X_train_processed = (
        fold_preprocessor
        .fit_transform(X_train)
        .astype(np.float32)
    )

    X_val_processed = (
        fold_preprocessor
        .transform(X_val)
        .astype(np.float32)
    )

    # standardize transformed targets
    target_mean = y_train_log.mean(
        axis=0,
        keepdims=True
    )

    target_std = y_train_log.std(
        axis=0,
        keepdims=True
    )

    target_std[target_std == 0] = 1.0

    y_train_scaled = (
        (y_train_log - target_mean)
        / target_std
    ).astype(np.float32)

    y_val_scaled = (
        (y_val_log - target_mean)
        / target_std
    ).astype(np.float32)

    input_dim = X_train_processed.shape[1]

    print("Processed predictors:", input_dim)

    # train each architecture
    for config_id, result in (
        expanded_mlp_config_results.items()
    ):

        config = result["config"]
        start_time = time.time()

        print(
            f"\nConfig {config_id}/"
            f"{len(expanded_mlp_configs)}"
        )

        print(config["name"])

        # reset random seeds
        model_seed = SEED + fold

        random.seed(model_seed)
        np.random.seed(model_seed)
        torch.manual_seed(model_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(model_seed)

        # create training loader
        train_generator = torch.Generator()
        train_generator.manual_seed(model_seed)

        train_dataset = TensorDataset(
            torch.from_numpy(X_train_processed),
            torch.from_numpy(y_train_scaled)
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=DL_BATCH_SIZE,
            shuffle=True,
            generator=train_generator,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        # create model
        model = MLP(
            input_dim=input_dim,
            output_dim=len(TARGETS),
            hidden_layers=config["hidden_layers"],
            dropout=config["dropout"],
            activation=config["activation"]
        ).to(DEVICE)

        criterion = nn.MSELoss()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=DL_LEARNING_RATE,
            weight_decay=0.0
        )

        # reduce learning rate after a plateau
        scheduler = (
            torch.optim.lr_scheduler
            .ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=0.5,
                patience=8,
                min_lr=1e-5
            )
        )

        best_val_smape = np.inf
        best_epoch = 0
        best_state = None

        # train for all 200 epochs
        for epoch in range(
            1,
            DL_MAX_EPOCHS + 1
        ):

            model.train()
            total_train_loss = 0.0

            for batch_x, batch_y in train_loader:

                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)

                optimizer.zero_grad()

                batch_predictions = model(batch_x)

                loss = criterion(
                    batch_predictions,
                    batch_y
                )

                loss.backward()
                optimizer.step()

                total_train_loss += (
                    loss.item()
                    * len(batch_x)
                )

            train_loss = (
                total_train_loss
                / len(train_dataset)
            )

            # calculate validation predictions
            val_predictions_scaled = predict_dl(
                model,
                X_val_processed
            )

            val_loss = np.mean(
                (
                    val_predictions_scaled
                    - y_val_scaled
                ) ** 2
            )

            scheduler.step(val_loss)

            # return to original units
            val_predictions_log = (
                val_predictions_scaled
                * target_std
                + target_mean
            )

            val_predictions = inverse_signed_log1p(
                val_predictions_log
            )

            val_predictions = apply_dl_accounting(
                val_predictions,
                X_val
            )

            val_smape = evaluate_predictions(
                y_val_original.to_numpy(),
                val_predictions
            )["AVERAGE"]

            # save best checkpoint
            if val_smape < best_val_smape:

                best_val_smape = val_smape
                best_epoch = epoch

                best_state = copy.deepcopy(
                    model.state_dict()
                )

            # print progress
            if (
                epoch == 1
                or epoch % 10 == 0
                or epoch == DL_MAX_EPOCHS
            ):

                train_predictions_scaled = predict_dl(
                    model,
                    X_train_processed
                )

                train_predictions_log = (
                    train_predictions_scaled
                    * target_std
                    + target_mean
                )

                train_predictions = (
                    inverse_signed_log1p(
                        train_predictions_log
                    )
                )

                train_predictions = apply_dl_accounting(
                    train_predictions,
                    X_train
                )

                train_smape = evaluate_predictions(
                    y_train_original.to_numpy(),
                    train_predictions
                )["AVERAGE"]

                current_lr = (
                    optimizer.param_groups[0]["lr"]
                )

                print(
                    f"Epoch {epoch:03d} | "
                    f"train loss={train_loss:.4f} | "
                    f"val loss={val_loss:.4f} | "
                    f"train sMAPE={train_smape:.4f} | "
                    f"val sMAPE={val_smape:.4f} | "
                    f"best={best_val_smape:.4f} | "
                    f"lr={current_lr:.6f}"
                )

        # restore best checkpoint
        model.load_state_dict(best_state)

        # generate best fold predictions
        fold_predictions_scaled = predict_dl(
            model,
            X_val_processed
        )

        fold_predictions_log = (
            fold_predictions_scaled
            * target_std
            + target_mean
        )

        fold_predictions = inverse_signed_log1p(
            fold_predictions_log
        )

        fold_predictions = apply_dl_accounting(
            fold_predictions,
            X_val
        )

        # store OOF predictions
        result["oof_predictions"][
            val_idx
        ] = fold_predictions

        fold_score = evaluate_predictions(
            y_val_original.to_numpy(),
            fold_predictions
        )["AVERAGE"]

        result["fold_scores"].append(
            fold_score
        )

        result["best_epochs"].append(
            best_epoch
        )

        elapsed = time.time() - start_time

        print(
            f"Best fold sMAPE={fold_score:.4f} | "
            f"best epoch={best_epoch} | "
            f"time={elapsed / 60:.1f} minutes"
        )

        del model
        del optimizer
        del scheduler

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# summarize architectures
expanded_mlp_results = []

for config_id, result in (
    expanded_mlp_config_results.items()
):

    expanded_mlp_results.append({
        "config_id": config_id,
        "model": result["config"]["name"],
        "mean_cv_smape": np.mean(
            result["fold_scores"]
        ),
        "std_cv_smape": np.std(
            result["fold_scores"],
            ddof=1
        ),
        "mean_best_epoch": np.mean(
            result["best_epochs"]
        )
    })

expanded_mlp_results = (
    pd.DataFrame(expanded_mlp_results)
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(expanded_mlp_results)


# select best architecture
best_expanded_config_id = int(
    expanded_mlp_results.loc[
        0,
        "config_id"
    ]
)

best_expanded_result = (
    expanded_mlp_config_results[
        best_expanded_config_id
    ]
)

best_expanded_config = (
    best_expanded_result["config"]
)

best_expanded_mlp_oof = (
    best_expanded_result[
        "oof_predictions"
    ]
)

best_expanded_scores = evaluate_predictions(
    y.to_numpy(),
    best_expanded_mlp_oof
)

print(
    "Best expanded MLP:",
    best_expanded_config["name"]
)

print(
    f"CV sMAPE: "
    f"{expanded_mlp_results.loc[0, 'mean_cv_smape']:.4f} ± "
    f"{expanded_mlp_results.loc[0, 'std_cv_smape']:.4f}"
)


# display target results
best_expanded_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        best_expanded_scores[target]
        for target in TARGETS
    ] + [
        best_expanded_scores["AVERAGE"]
    ]
})

display(best_expanded_target_results)


Fold 1/5
Processed predictors: 332

Config 1/3
Reference Deep MLP
Epoch 001 | train loss=0.4575 | val loss=0.2462 | train sMAPE=54.3904 | val sMAPE=54.8250 | best=54.8250 | lr=0.001000
Epoch 010 | train loss=0.1811 | val loss=0.1471 | train sMAPE=39.1851 | val sMAPE=39.5192 | best=39.5192 | lr=0.001000
Epoch 020 | train loss=0.1570 | val loss=0.1358 | train sMAPE=36.0435 | val sMAPE=36.3696 | best=35.7573 | lr=0.001000
Epoch 030 | train loss=0.1484 | val loss=0.1316 | train sMAPE=35.1440 | val sMAPE=35.4359 | best=33.7625 | lr=0.001000
Epoch 040 | train loss=0.1386 | val loss=0.1274 | train sMAPE=34.6970 | val sMAPE=35.0206 | best=32.9220 | lr=0.001000
Epoch 050 | train loss=0.1344 | val loss=0.1269 | train sMAPE=32.8606 | val sMAPE=33.2009 | best=32.5512 | lr=0.001000
Epoch 060 | train loss=0.1307 | val loss=0.1222 | train sMAPE=33.6250 | val sMAPE=33.9587 | best=31.9750 | lr=0.001000
Epoch 070 | train loss=0.1211 | val loss=0.1128 | train sMAPE=30.8217 | val sMAPE=31.2169 | best=30.

,config_id,model,mean_cv_smape,std_cv_smape,mean_best_epoch
0,2,Wider Deep MLP,26.5376,0.3040,157.6000
1,1,Reference Deep MLP,28.6189,0.4054,168.2000
2,3,Deeper MLP,28.8613,0.3181,170.6000


Best expanded MLP: Wider Deep MLP
CV sMAPE: 26.5376 ± 0.3040


,target,smape
0,Q0_TOTAL_ASSETS,17.7710
1,Q0_TOTAL_LIABILITIES,21.8991
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,24.4149
3,Q0_GROSS_PROFIT,28.9624
4,Q0_COST_OF_REVENUES,22.0408
5,Q0_REVENUES,14.6823
6,Q0_OPERATING_INCOME,49.5384
7,Q0_OPERATING_EXPENSES,11.2311
8,Q0_EBITDA,48.2984
9,AVERAGE,26.5376


The **Wider Deep MLP** achieved the best deep-learning performance, with a mean five-fold CV sMAPE of **26.5376 ± 0.3040**. It outperformed both the reference Deep MLP (**28.6189**) and the Deeper MLP (**28.8613**). This suggests that increasing the network's width was more useful than adding additional layers.

The learning-rate scheduler and best-model checkpointing also improved performance. The scheduler reduced the learning rate when validation loss stopped improving, allowing the models to make smaller adjustments later in training. The best epochs were generally between **158 and 171**, confirming that the earlier patience of 10 epochs would have stopped training too soon.

Although the Wider Deep MLP produced stable results across folds, its sMAPE remained higher than the final non-deep-learning ensemble. The largest errors continued to come from **Q0_OPERATING_INCOME** and **Q0_EBITDA**, while revenue, operating expenses, assets, and liabilities were predicted more accurately.

Overall, the results show that:

- Increasing network width improved performance more than increasing depth
- A learning-rate scheduler supported longer and more stable training
- Saving the best validation checkpoint was more reliable than using the final epoch
- Accounting-derived targets remain the main source of prediction error
- The deep-learning model has not yet outperformed the non-deep-learning ensemble

Before selecting the final deep-learning model, we will test a **hybrid GRU–MLP model**. The GRU branch will learn patterns across the ten historical quarters, while the MLP branch will process the remaining static predictors.

### Hybrid GRU-MLP Model

In [9]:
import copy
import random
import time
import torch
import torch.nn as nn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# set device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# set random seed
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# define quarterly financial measures
SEQUENCE_BASES = [
    "TOTAL_ASSETS",
    "TOTAL_CURRENT_ASSETS",
    "TOTAL_NONCURRENT_ASSETS",
    "TOTAL_LIABILITIES",
    "TOTAL_CURRENT_LIABILITIES",
    "TOTAL_NONCURRENT_LIABILITIES",
    "TOTAL_LIABILITIES_AND_EQUITY",
    "TOTAL_STOCKHOLDERS_EQUITY",
    "NET_INCOME",
    "GROSS_PROFIT",
    "COST_OF_REVENUES",
    "REVENUES",
    "OPERATING_INCOME",
    "OPERATING_EXPENSES",
    "EBITDA",
    "DEPRECIATION_AND_AMORTIZATION"
]

# arrange quarters from oldest to most recent
QUARTERS = list(range(10, 0, -1))

sequence_columns = [
    f"Q{quarter}_{variable}"
    for quarter in QUARTERS
    for variable in SEQUENCE_BASES
]

missing_sequence_columns = [
    column
    for column in sequence_columns
    if column not in X.columns
]

assert not missing_sequence_columns, (
    f"Missing sequence columns: {missing_sequence_columns}"
)

static_columns = [
    column
    for column in X.columns
    if column not in sequence_columns
]

print("Sequence predictors:", len(sequence_columns))
print("Static predictors:", len(static_columns))
print(
    "GRU input shape:",
    f"10 quarters × {len(SEQUENCE_BASES)} measures"
)

# create static preprocessing pipeline
def create_static_preprocessor(X_fold):

    numeric_static = (
        X_fold[static_columns]
        .select_dtypes(include=np.number)
        .columns
        .tolist()
    )

    categorical_static = [
        column
        for column in static_columns
        if column not in numeric_static
    ]

    transformers = []

    if numeric_static:
        transformers.append(
            (
                "numeric",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]),
                numeric_static
            )
        )

    if categorical_static:
        transformers.append(
            (
                "categorical",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value="MISSING"
                        )
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                            dtype=np.float32
                        )
                    )
                ]),
                categorical_static
            )
        )

    return ColumnTransformer(
        transformers,
        remainder="drop"
    )


# prepare quarterly sequences
def prepare_sequence_data(
    X_train_fold,
    X_val_fold
):

    n_measures = len(SEQUENCE_BASES)

    train_sequence = (
        X_train_fold[sequence_columns]
        .to_numpy(dtype=np.float32)
        .reshape(-1, 10, n_measures)
    )

    val_sequence = (
        X_val_fold[sequence_columns]
        .to_numpy(dtype=np.float32)
        .reshape(-1, 10, n_measures)
    )

    # fit imputation using the training fold
    sequence_imputer = SimpleImputer(
        strategy="median"
    )

    train_flat = train_sequence.reshape(
        -1,
        n_measures
    )

    val_flat = val_sequence.reshape(
        -1,
        n_measures
    )

    train_flat = sequence_imputer.fit_transform(
        train_flat
    )

    val_flat = sequence_imputer.transform(
        val_flat
    )

    # scale each financial measure
    sequence_scaler = StandardScaler()

    train_flat = sequence_scaler.fit_transform(
        train_flat
    )

    val_flat = sequence_scaler.transform(
        val_flat
    )

    train_sequence = train_flat.reshape(
        -1,
        10,
        n_measures
    ).astype(np.float32)

    val_sequence = val_flat.reshape(
        -1,
        10,
        n_measures
    ).astype(np.float32)

    return (
        train_sequence,
        val_sequence,
        sequence_imputer,
        sequence_scaler
    )


# define hybrid GRU-MLP model
class HybridGRUMLP(nn.Module):

    def __init__(
        self,
        sequence_features,
        static_features,
        output_features
    ):
        super().__init__()

        # quarterly sequence branch
        self.gru = nn.GRU(
            input_size=sequence_features,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        self.sequence_dropout = nn.Dropout(0.10)

        # static predictor branch
        self.static_branch = nn.Sequential(
            nn.Linear(static_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.10)
        )

        # combined branch
        self.combined_branch = nn.Sequential(
            nn.Linear(64 + 128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, output_features)
        )

        # initialize linear layers
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(
                    layer.weight,
                    nonlinearity="relu"
                )
                nn.init.zeros_(layer.bias)

    def forward(
        self,
        sequence_input,
        static_input
    ):

        _, hidden_state = self.gru(
            sequence_input
        )

        sequence_summary = (
            hidden_state[-1]
        )

        sequence_summary = (
            self.sequence_dropout(
                sequence_summary
            )
        )

        static_summary = self.static_branch(
            static_input
        )

        combined = torch.cat(
            [
                sequence_summary,
                static_summary
            ],
            dim=1
        )

        return self.combined_branch(
            combined
        )


# generate model predictions
def predict_hybrid(
    model,
    sequence_data,
    static_data,
    batch_size=2048
):

    dataset = TensorDataset(
        torch.tensor(
            sequence_data,
            dtype=torch.float32
        ),
        torch.tensor(
            static_data,
            dtype=torch.float32
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    predictions = []

    model.eval()

    with torch.no_grad():

        for sequence_batch, static_batch in loader:

            batch_predictions = model(
                sequence_batch.to(device),
                static_batch.to(device)
            )

            predictions.append(
                batch_predictions
                .cpu()
                .numpy()
            )

    return np.vstack(predictions)


# apply accounting relationships
def apply_gru_accounting(
    predictions,
    X_fold
):

    adjusted = predictions.copy()

    target_index = {
        target: i
        for i, target in enumerate(TARGETS)
    }

    revenue_index = target_index[
        "Q0_REVENUES"
    ]

    cost_index = target_index[
        "Q0_COST_OF_REVENUES"
    ]

    gross_profit_index = target_index[
        "Q0_GROSS_PROFIT"
    ]

    operating_expense_index = target_index[
        "Q0_OPERATING_EXPENSES"
    ]

    operating_income_index = target_index[
        "Q0_OPERATING_INCOME"
    ]

    ebitda_index = target_index[
        "Q0_EBITDA"
    ]

    # derive gross profit
    adjusted[:, gross_profit_index] = (
        adjusted[:, revenue_index]
        - adjusted[:, cost_index]
    )

    # derive operating income
    adjusted[:, operating_income_index] = (
        adjusted[:, gross_profit_index]
        - adjusted[:, operating_expense_index]
    )

    # derive EBITDA using Q1 D&A
    q1_da = X_fold[
        "Q1_DEPRECIATION_AND_AMORTIZATION"
    ].to_numpy(dtype=float)

    adjusted[:, ebitda_index] = (
        adjusted[:, operating_income_index]
        + q1_da
    )

    return adjusted


# define training settings
MAX_EPOCHS = 200
BATCH_SIZE = 1024
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0
PRINT_EVERY = 10

# initialize results
gru_oof = np.zeros(
    (len(y), len(TARGETS)),
    dtype=float
)

gru_fold_scores = []
gru_best_epochs = []

# train with five-fold cross-validation
for fold, (train_idx, val_idx) in enumerate(
    CV_SPLITS,
    start=1
):

    print(f"\n{'=' * 60}")
    print(f"Fold {fold}/{N_FOLDS}")
    print(f"{'=' * 60}")

    set_seed(SEED + fold)

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_transformed = (
        y_transformed.iloc[train_idx]
        .to_numpy(dtype=np.float32)
    )

    y_val_original = (
        y.iloc[val_idx]
        .to_numpy(dtype=float)
    )

    # prepare sequence predictors
    (
        train_sequence,
        val_sequence,
        sequence_imputer,
        sequence_scaler
    ) = prepare_sequence_data(
        X_train,
        X_val
    )

    # prepare static predictors
    static_preprocessor = (
        create_static_preprocessor(
            X_train
        )
    )

    train_static = (
        static_preprocessor
        .fit_transform(
            X_train[static_columns]
        )
        .astype(np.float32)
    )

    val_static = (
        static_preprocessor
        .transform(
            X_val[static_columns]
        )
        .astype(np.float32)
    )

    # standardize transformed targets
    target_mean = (
        y_train_transformed
        .mean(axis=0)
    )

    target_std = (
        y_train_transformed
        .std(axis=0)
    )

    target_std[target_std == 0] = 1.0

    y_train_scaled = (
        (
            y_train_transformed
            - target_mean
        )
        / target_std
    ).astype(np.float32)

    # create training loader
    train_dataset = TensorDataset(
        torch.tensor(
            train_sequence,
            dtype=torch.float32
        ),
        torch.tensor(
            train_static,
            dtype=torch.float32
        ),
        torch.tensor(
            y_train_scaled,
            dtype=torch.float32
        )
    )

    train_generator = torch.Generator()
    train_generator.manual_seed(
        SEED + fold
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator
    )

    # initialize model
    model = HybridGRUMLP(
        sequence_features=len(SEQUENCE_BASES),
        static_features=train_static.shape[1],
        output_features=len(TARGETS)
    ).to(device)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=8,
            min_lr=1e-5
        )
    )

    best_score = np.inf
    best_epoch = None
    best_state = None

    fold_start = time.time()

    # train for all epochs
    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        total_train_loss = 0.0

        for (
            sequence_batch,
            static_batch,
            target_batch
        ) in train_loader:

            sequence_batch = (
                sequence_batch.to(device)
            )

            static_batch = (
                static_batch.to(device)
            )

            target_batch = (
                target_batch.to(device)
            )

            optimizer.zero_grad()

            predictions = model(
                sequence_batch,
                static_batch
            )

            loss = loss_function(
                predictions,
                target_batch
            )

            loss.backward()
            optimizer.step()

            total_train_loss += (
                loss.item()
                * len(target_batch)
            )

        train_loss = (
            total_train_loss
            / len(train_dataset)
        )

        # calculate validation predictions
        val_predictions_scaled = predict_hybrid(
            model,
            val_sequence,
            val_static
        )

        val_predictions_transformed = (
            val_predictions_scaled
            * target_std
            + target_mean
        )

        val_predictions = (
            inverse_signed_log1p(
                val_predictions_transformed
            )
        )

        val_predictions = apply_gru_accounting(
            val_predictions,
            X_val
        )

        val_score = evaluate_predictions(
            y_val_original,
            val_predictions
        )["AVERAGE"]

        # use validation sMAPE for scheduling
        scheduler.step(val_score)

        # save best validation checkpoint
        if val_score < best_score:

            best_score = val_score
            best_epoch = epoch

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

        # print training progress
        if (
            epoch == 1
            or epoch % PRINT_EVERY == 0
            or epoch == MAX_EPOCHS
        ):

            current_lr = (
                optimizer
                .param_groups[0]["lr"]
            )

            elapsed = (
                time.time() - fold_start
            ) / 60

            print(
                f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
                f"train loss={train_loss:.4f} | "
                f"validation sMAPE={val_score:.4f} | "
                f"best={best_score:.4f} | "
                f"lr={current_lr:.6f} | "
                f"time={elapsed:.1f} min"
            )

    # restore best checkpoint
    model.load_state_dict(best_state)
    model.to(device)

    # generate best fold predictions
    best_predictions_scaled = predict_hybrid(
        model,
        val_sequence,
        val_static
    )

    best_predictions_transformed = (
        best_predictions_scaled
        * target_std
        + target_mean
    )

    best_predictions = inverse_signed_log1p(
        best_predictions_transformed
    )

    best_predictions = apply_gru_accounting(
        best_predictions,
        X_val
    )

    gru_oof[val_idx] = best_predictions
    gru_fold_scores.append(best_score)
    gru_best_epochs.append(best_epoch)

    fold_minutes = (
        time.time() - fold_start
    ) / 60

    print(
        f"\nFold {fold} complete | "
        f"best epoch={best_epoch} | "
        f"sMAPE={best_score:.4f} | "
        f"time={fold_minutes:.1f} minutes"
    )

# calculate final CV results
gru_scores = evaluate_predictions(
    y.to_numpy(),
    gru_oof
)

gru_cv_mean = np.mean(
    gru_fold_scores
)

gru_cv_std = np.std(
    gru_fold_scores,
    ddof=1
)

# display model summary
gru_summary = pd.DataFrame({
    "model": ["Hybrid GRU-MLP"],
    "mean_cv_smape": [gru_cv_mean],
    "std_cv_smape": [gru_cv_std],
    "mean_best_epoch": [
        np.mean(gru_best_epochs)
    ]
})

# display target results
gru_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        gru_scores[target]
        for target in TARGETS
    ] + [gru_scores["AVERAGE"]]
})

display(gru_summary)
display(gru_target_results)

Device: cpu
Sequence predictors: 160
Static predictors: 42
GRU input shape: 10 quarters × 16 measures

Fold 1/5
Epoch   1/200 | train loss=0.5983 | validation sMAPE=69.2634 | best=69.2634 | lr=0.001000 | time=0.1 min
Epoch  10/200 | train loss=0.1853 | validation sMAPE=42.8700 | best=42.8700 | lr=0.001000 | time=0.7 min
Epoch  20/200 | train loss=0.1589 | validation sMAPE=35.7625 | best=35.7625 | lr=0.001000 | time=1.5 min
Epoch  30/200 | train loss=0.1470 | validation sMAPE=37.7907 | best=32.2295 | lr=0.001000 | time=2.4 min
Epoch  40/200 | train loss=0.1391 | validation sMAPE=31.2840 | best=31.2840 | lr=0.001000 | time=3.1 min
Epoch  50/200 | train loss=0.1350 | validation sMAPE=34.8928 | best=31.0601 | lr=0.001000 | time=3.9 min
Epoch  60/200 | train loss=0.1310 | validation sMAPE=32.1839 | best=30.3075 | lr=0.001000 | time=4.7 min
Epoch  70/200 | train loss=0.1234 | validation sMAPE=30.1522 | best=29.3201 | lr=0.000500 | time=5.5 min
Epoch  80/200 | train loss=0.1209 | validation s

,model,mean_cv_smape,std_cv_smape,mean_best_epoch
0,Hybrid GRU-MLP,28.7174,0.5319,162.4000


,target,smape
0,Q0_TOTAL_ASSETS,20.5797
1,Q0_TOTAL_LIABILITIES,24.1946
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,29.3069
3,Q0_GROSS_PROFIT,30.7779
4,Q0_COST_OF_REVENUES,24.0608
5,Q0_REVENUES,15.2991
6,Q0_OPERATING_INCOME,51.8739
7,Q0_OPERATING_EXPENSES,11.7890
8,Q0_EBITDA,50.5745
9,AVERAGE,28.7174


The Hybrid GRU–MLP achieved a mean five-fold CV sMAPE of **28.7174 ± 0.5319**. This was worse than the Wider Deep MLP, which achieved **26.5376 ± 0.3040**. The hybrid model also produced a higher sMAPE for every individual target.

The GRU branch was designed to learn patterns across the ten historical quarters, while the MLP branch processed the remaining static predictors. However, the sequential representation did not improve prediction accuracy. One possible explanation is that the standard MLP already received all quarterly predictors directly. Compressing these predictors into a 64-unit GRU summary may have removed useful information. The sequence of ten quarters may also be too short for the GRU to provide a meaningful advantage.

The mean best epoch was **162.4**, which again supports the use of a learning-rate scheduler and longer training. It also confirms that the earlier patience-based stopping rule would likely have stopped training too soon.

Overall, the experiment shows that:

- The Wider Deep MLP remains the strongest deep-learning architecture
- The GRU did not improve any of the nine target predictions
- The Hybrid GRU–MLP was less accurate and slightly less stable across folds
- Operating income and EBITDA remain the most difficult targets
- Further GRU architecture tuning is unlikely to justify its additional complexity and training time

### Next Step

The next step is to refine the Wider Deep MLP by testing **activation function** and **weight decay**. The current model uses ReLU without weight decay. We will compare ReLU and Leaky ReLU, both with and without a weight decay of $10^{-4}$.

This controlled experiment will determine whether allowing small gradients for negative activations or adding regularization can improve the accuracy and stability of our strongest deep-learning model. 

In [10]:
import copy
import random
import time
import torch
import torch.nn as nn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# set device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# set random seed
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# create preprocessing pipeline
def create_tuning_preprocessor(X_fold):

    numeric_columns = (
        X_fold
        .select_dtypes(include=np.number)
        .columns
        .tolist()
    )

    categorical_columns = (
        X_fold
        .select_dtypes(exclude=np.number)
        .columns
        .tolist()
    )

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True
                    )
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="MISSING"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                        dtype=np.float32
                    )
                )
            ]),
            categorical_columns
        )
    ])


# define wider deep MLP
class TunedWiderMLP(nn.Module):

    def __init__(
        self,
        input_size,
        output_size,
        activation
    ):
        super().__init__()

        hidden_sizes = [
            1024,
            512,
            256,
            128
        ]

        layers = []
        previous_size = input_size

        for hidden_size in hidden_sizes:

            layers.append(
                nn.Linear(
                    previous_size,
                    hidden_size
                )
            )

            layers.append(
                nn.BatchNorm1d(hidden_size)
            )

            if activation == "relu":
                layers.append(nn.ReLU())

            elif activation == "leaky_relu":
                layers.append(
                    nn.LeakyReLU(
                        negative_slope=0.01
                    )
                )

            layers.append(
                nn.Dropout(0.10)
            )

            previous_size = hidden_size

        layers.append(
            nn.Linear(
                previous_size,
                output_size
            )
        )

        self.network = nn.Sequential(
            *layers
        )

        # initialize linear layers
        for layer in self.modules():

            if isinstance(layer, nn.Linear):

                nn.init.kaiming_normal_(
                    layer.weight,
                    nonlinearity="leaky_relu"
                    if activation == "leaky_relu"
                    else "relu"
                )

                nn.init.zeros_(layer.bias)

    def forward(self, predictors):
        return self.network(predictors)


# generate predictions
def predict_tuned_mlp(
    model,
    predictors,
    targets=None,
    batch_size=2048
):

    predictor_tensor = torch.tensor(
        predictors,
        dtype=torch.float32
    )

    if targets is None:

        dataset = TensorDataset(
            predictor_tensor
        )

    else:

        target_tensor = torch.tensor(
            targets,
            dtype=torch.float32
        )

        dataset = TensorDataset(
            predictor_tensor,
            target_tensor
        )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    predictions = []
    total_loss = 0.0
    loss_function = nn.MSELoss()

    model.eval()

    with torch.no_grad():

        for batch in loader:

            batch_predictors = batch[0].to(
                device
            )

            batch_predictions = model(
                batch_predictors
            )

            predictions.append(
                batch_predictions
                .cpu()
                .numpy()
            )

            if targets is not None:

                batch_targets = batch[1].to(
                    device
                )

                batch_loss = loss_function(
                    batch_predictions,
                    batch_targets
                )

                total_loss += (
                    batch_loss.item()
                    * len(batch_targets)
                )

    predictions = np.vstack(predictions)

    if targets is None:
        return predictions

    average_loss = (
        total_loss / len(dataset)
    )

    return predictions, average_loss


# apply accounting relationships
def apply_tuning_accounting(
    predictions,
    X_fold
):

    adjusted = predictions.copy()

    target_index = {
        target: index
        for index, target
        in enumerate(TARGETS)
    }

    revenue_index = target_index[
        "Q0_REVENUES"
    ]

    cost_index = target_index[
        "Q0_COST_OF_REVENUES"
    ]

    gross_profit_index = target_index[
        "Q0_GROSS_PROFIT"
    ]

    operating_expense_index = target_index[
        "Q0_OPERATING_EXPENSES"
    ]

    operating_income_index = target_index[
        "Q0_OPERATING_INCOME"
    ]

    ebitda_index = target_index[
        "Q0_EBITDA"
    ]

    # derive gross profit
    adjusted[:, gross_profit_index] = (
        adjusted[:, revenue_index]
        - adjusted[:, cost_index]
    )

    # derive operating income
    adjusted[:, operating_income_index] = (
        adjusted[:, gross_profit_index]
        - adjusted[:, operating_expense_index]
    )

    # derive EBITDA
    q1_da = X_fold[
        "Q1_DEPRECIATION_AND_AMORTIZATION"
    ].to_numpy(dtype=float)

    adjusted[:, ebitda_index] = (
        adjusted[:, operating_income_index]
        + q1_da
    )

    return adjusted


# define new configurations
tuning_configs = [
    {
        "name": "Leaky ReLU - No Weight Decay",
        "activation": "leaky_relu",
        "weight_decay": 0.0
    },
    {
        "name": "ReLU - Weight Decay",
        "activation": "relu",
        "weight_decay": 1e-4
    },
    {
        "name": "Leaky ReLU - Weight Decay",
        "activation": "leaky_relu",
        "weight_decay": 1e-4
    }
]

# define training settings
MAX_EPOCHS = 200
BATCH_SIZE = 1024
LEARNING_RATE = 0.001
PRINT_EVERY = 10

# initialize results
tuning_config_results = {
    config_id: {
        "params": config,
        "oof_predictions": np.zeros(
            (len(y), len(TARGETS))
        ),
        "fold_scores": [],
        "best_epochs": []
    }
    for config_id, config in enumerate(
        tuning_configs,
        start=1
    )
}

# train with five-fold cross-validation
for fold, (train_idx, val_idx) in enumerate(
    CV_SPLITS,
    start=1
):

    print(f"\n{'=' * 65}")
    print(f"Fold {fold}/{N_FOLDS}")
    print(f"{'=' * 65}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train_transformed = (
        y_transformed.iloc[train_idx]
        .to_numpy(dtype=np.float32)
    )

    y_val_transformed = (
        y_transformed.iloc[val_idx]
        .to_numpy(dtype=np.float32)
    )

    y_val_original = (
        y.iloc[val_idx]
        .to_numpy(dtype=float)
    )

    # preprocess predictors
    tuning_preprocessor = (
        create_tuning_preprocessor(
            X_train
        )
    )

    X_train_processed = (
        tuning_preprocessor
        .fit_transform(X_train)
        .astype(np.float32)
    )

    X_val_processed = (
        tuning_preprocessor
        .transform(X_val)
        .astype(np.float32)
    )

    print(
        "Processed predictors:",
        X_train_processed.shape[1]
    )

    # standardize transformed targets
    target_mean = (
        y_train_transformed
        .mean(axis=0)
    )

    target_std = (
        y_train_transformed
        .std(axis=0)
    )

    target_std[target_std == 0] = 1.0

    y_train_scaled = (
        (
            y_train_transformed
            - target_mean
        )
        / target_std
    ).astype(np.float32)

    y_val_scaled = (
        (
            y_val_transformed
            - target_mean
        )
        / target_std
    ).astype(np.float32)

    for config_id, result in (
        tuning_config_results.items()
    ):

        config = result["params"]

        print(
            f"\nConfig {config_id}/"
            f"{len(tuning_configs)} | "
            f"{config['name']}"
        )

        # reset seed for fair comparison
        set_seed(SEED + fold)

        train_dataset = TensorDataset(
            torch.tensor(
                X_train_processed,
                dtype=torch.float32
            ),
            torch.tensor(
                y_train_scaled,
                dtype=torch.float32
            )
        )

        train_generator = torch.Generator()
        train_generator.manual_seed(
            SEED + fold
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=train_generator
        )

        # initialize model
        model = TunedWiderMLP(
            input_size=X_train_processed.shape[1],
            output_size=len(TARGETS),
            activation=config["activation"]
        ).to(device)

        loss_function = nn.MSELoss()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=config[
                "weight_decay"
            ]
        )

        scheduler = (
            torch.optim.lr_scheduler
            .ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=0.5,
                patience=8,
                min_lr=1e-5
            )
        )

        best_score = np.inf
        best_epoch = None
        best_state = None

        config_start = time.time()

        # train model
        for epoch in range(
            1,
            MAX_EPOCHS + 1
        ):

            model.train()
            total_train_loss = 0.0

            for (
                predictor_batch,
                target_batch
            ) in train_loader:

                predictor_batch = (
                    predictor_batch.to(device)
                )

                target_batch = (
                    target_batch.to(device)
                )

                optimizer.zero_grad()

                batch_predictions = model(
                    predictor_batch
                )

                loss = loss_function(
                    batch_predictions,
                    target_batch
                )

                loss.backward()
                optimizer.step()

                total_train_loss += (
                    loss.item()
                    * len(target_batch)
                )

            train_loss = (
                total_train_loss
                / len(train_dataset)
            )

            # evaluate validation data
            (
                val_predictions_scaled,
                val_loss
            ) = predict_tuned_mlp(
                model,
                X_val_processed,
                y_val_scaled
            )

            val_predictions_transformed = (
                val_predictions_scaled
                * target_std
                + target_mean
            )

            val_predictions = (
                inverse_signed_log1p(
                    val_predictions_transformed
                )
            )

            val_predictions = (
                apply_tuning_accounting(
                    val_predictions,
                    X_val
                )
            )

            val_score = evaluate_predictions(
                y_val_original,
                val_predictions
            )["AVERAGE"]

            # update learning rate
            scheduler.step(val_loss)

            # save best checkpoint
            if val_score < best_score:

                best_score = val_score
                best_epoch = epoch

                best_state = {
                    key: value
                    .detach()
                    .cpu()
                    .clone()
                    for key, value
                    in model.state_dict().items()
                }

            # print progress
            if (
                epoch == 1
                or epoch % PRINT_EVERY == 0
                or epoch == MAX_EPOCHS
            ):

                current_lr = (
                    optimizer
                    .param_groups[0]["lr"]
                )

                elapsed = (
                    time.time()
                    - config_start
                ) / 60

                print(
                    f"Epoch {epoch:3d} | "
                    f"train loss={train_loss:.4f} | "
                    f"val loss={val_loss:.4f} | "
                    f"val sMAPE={val_score:.4f} | "
                    f"best={best_score:.4f} | "
                    f"lr={current_lr:.6f} | "
                    f"time={elapsed:.1f} min"
                )

        # restore best checkpoint
        model.load_state_dict(best_state)
        model.to(device)

        best_predictions_scaled = (
            predict_tuned_mlp(
                model,
                X_val_processed
            )
        )

        best_predictions_transformed = (
            best_predictions_scaled
            * target_std
            + target_mean
        )

        best_predictions = (
            inverse_signed_log1p(
                best_predictions_transformed
            )
        )

        best_predictions = (
            apply_tuning_accounting(
                best_predictions,
                X_val
            )
        )

        # store fold results
        result["oof_predictions"][
            val_idx
        ] = best_predictions

        result["fold_scores"].append(
            best_score
        )

        result["best_epochs"].append(
            best_epoch
        )

        elapsed = (
            time.time()
            - config_start
        ) / 60

        print(
            f"Best fold sMAPE={best_score:.4f} | "
            f"best epoch={best_epoch} | "
            f"time={elapsed:.1f} minutes"
        )

        del model
        torch.cuda.empty_cache()

# summarize new configurations
tuning_results = []

for config_id, result in (
    tuning_config_results.items()
):

    tuning_results.append({
        "config_id": config_id,
        "model": result["params"]["name"],
        "activation": result[
            "params"
        ]["activation"],
        "weight_decay": result[
            "params"
        ]["weight_decay"],
        "mean_cv_smape": np.mean(
            result["fold_scores"]
        ),
        "std_cv_smape": np.std(
            result["fold_scores"],
            ddof=1
        ),
        "mean_best_epoch": np.mean(
            result["best_epochs"]
        )
    })

tuning_results = pd.DataFrame(
    tuning_results
)

# add existing reference result
reference_result = pd.DataFrame([{
    "config_id": 0,
    "model": "ReLU - No Weight Decay",
    "activation": "relu",
    "weight_decay": 0.0,
    "mean_cv_smape": 26.5376,
    "std_cv_smape": 0.3040,
    "mean_best_epoch": 157.6
}])

activation_weight_decay_results = (
    pd.concat(
        [
            reference_result,
            tuning_results
        ],
        ignore_index=True
    )
    .sort_values("mean_cv_smape")
    .reset_index(drop=True)
)

display(activation_weight_decay_results)

# select the best new configuration
best_new_config_id = int(
    tuning_results
    .sort_values("mean_cv_smape")
    .iloc[0]["config_id"]
)

best_new_result = tuning_config_results[
    best_new_config_id
]

best_new_oof = best_new_result[
    "oof_predictions"
]

best_new_scores = evaluate_predictions(
    y.to_numpy(),
    best_new_oof
)

# display best new target results
best_new_target_results = pd.DataFrame({
    "target": TARGETS + ["AVERAGE"],
    "smape": [
        best_new_scores[target]
        for target in TARGETS
    ] + [best_new_scores["AVERAGE"]]
})

print(
    "Best overall configuration:",
    activation_weight_decay_results
    .loc[0, "model"]
)

display(best_new_target_results)


Fold 1/5
Processed predictors: 332

Config 1/3 | Leaky ReLU - No Weight Decay
Epoch   1 | train loss=0.4542 | val loss=0.2420 | val sMAPE=56.9698 | best=56.9698 | lr=0.001000 | time=0.1 min
Epoch  10 | train loss=0.1685 | val loss=0.1477 | val sMAPE=44.4763 | best=37.9264 | lr=0.001000 | time=1.1 min
Epoch  20 | train loss=0.1487 | val loss=0.1366 | val sMAPE=35.4433 | best=34.7903 | lr=0.001000 | time=2.3 min
Epoch  30 | train loss=0.1379 | val loss=0.1261 | val sMAPE=34.1413 | best=33.1003 | lr=0.001000 | time=3.5 min
Epoch  40 | train loss=0.1299 | val loss=0.1245 | val sMAPE=33.3450 | best=31.5427 | lr=0.001000 | time=4.7 min
Epoch  50 | train loss=0.1284 | val loss=0.1248 | val sMAPE=32.9734 | best=30.8382 | lr=0.001000 | time=6.0 min
Epoch  60 | train loss=0.1205 | val loss=0.1188 | val sMAPE=32.6308 | best=30.1709 | lr=0.001000 | time=7.2 min
Epoch  70 | train loss=0.1178 | val loss=0.1159 | val sMAPE=31.5259 | best=29.5393 | lr=0.001000 | time=8.6 min
Epoch  80 | train loss=0.

,config_id,model,activation,weight_decay,mean_cv_smape,std_cv_smape,mean_best_epoch
0,2,ReLU - Weight Decay,relu,0.0001,25.5476,0.6576,181.2000
1,1,Leaky ReLU - No Weight Decay,leaky_relu,0.0000,25.6478,0.5120,178.8000
2,3,Leaky ReLU - Weight Decay,leaky_relu,0.0001,25.7305,0.2183,178.8000
3,0,ReLU - No Weight Decay,relu,0.0000,26.5376,0.3040,157.6000


Best overall configuration: ReLU - Weight Decay


,target,smape
0,Q0_TOTAL_ASSETS,16.6015
1,Q0_TOTAL_LIABILITIES,20.5383
2,Q0_TOTAL_STOCKHOLDERS_EQUITY,23.6439
3,Q0_GROSS_PROFIT,28.1710
4,Q0_COST_OF_REVENUES,20.6823
5,Q0_REVENUES,13.2115
6,Q0_OPERATING_INCOME,49.1881
7,Q0_OPERATING_EXPENSES,10.0529
8,Q0_EBITDA,47.8385
9,AVERAGE,25.5476


Adding weight decay and changing the activation function improved the Wider Deep MLP compared with the original ReLU model.

The ReLU model with weight decay produced the lowest mean CV sMAPE of **25.5476**, while the Leaky ReLU model with weight decay achieved a very similar mean sMAPE of **25.7305**. The difference between them was only **0.1829**.

However, the Leaky ReLU model with weight decay had a much lower CV standard deviation:

- ReLU with weight decay: **25.5476 ± 0.6576**
- Leaky ReLU with weight decay: **25.7305 ± 0.2183**

Although its average sMAPE was slightly higher, the Leaky ReLU model produced much more consistent results across the five folds. This suggests that it is less sensitive to the particular training and validation samples used in each fold and may generalize more reliably to unseen data.

Weight decay helped control model complexity by discouraging excessively large weights, while Leaky ReLU maintained a small gradient for negative inputs. Combining these techniques provided the best balance between prediction accuracy and cross-validation stability. Therefore, we selected the **Wider Deep MLP with Leaky ReLU, 10% dropout, and weight decay of $10^{-4}$** as the final deep-learning model.

The selected model was then trained directly on the complete training dataset. Although the rounded mean best epoch from five-fold cross-validation was 179, we trained the final model for 200 epochs. The learning-rate scheduler reduces the learning rate as training stabilizes, so the final 21 epochs use relatively small parameter updates.

In [12]:
import joblib
import time
from pathlib import Path

# create output folder
DL_SAVE_DIRECTORY = ARTIFACT_DIR / "saved_dl_model"

DL_SAVE_DIRECTORY.mkdir(
    exist_ok=True
)

# define selected configuration
FINAL_DL_CONFIG = {
    "name": "Wider MLP - Leaky ReLU",
    "hidden_layers": [
        1024,
        512,
        256,
        128
    ],
    "activation": "leaky_relu",
    "dropout": 0.10,
    "weight_decay": 1e-4,
    "learning_rate": 0.001,
    "epochs": 200,
    "batch_size": 1024
}

set_seed(SEED)

print("Selected DL configuration:")
print(FINAL_DL_CONFIG)
print("Device:", device)

# fit preprocessing on all training data
final_dl_preprocessor = (
    create_tuning_preprocessor(X)
)

print("\nPreprocessing predictors...")

X_full_processed = (
    final_dl_preprocessor
    .fit_transform(X)
    .astype(np.float32)
)

X_test_processed = (
    final_dl_preprocessor
    .transform(X_test)
    .astype(np.float32)
)

print(
    "Processed training shape:",
    X_full_processed.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)

# standardize transformed targets
y_full_transformed = (
    y_transformed
    .to_numpy(dtype=np.float32)
)

final_target_mean = (
    y_full_transformed
    .mean(axis=0)
)

final_target_std = (
    y_full_transformed
    .std(axis=0)
)

final_target_std[
    final_target_std == 0
] = 1.0

y_full_scaled = (
    (
        y_full_transformed
        - final_target_mean
    )
    / final_target_std
).astype(np.float32)

# create training loader
final_train_dataset = TensorDataset(
    torch.tensor(
        X_full_processed,
        dtype=torch.float32
    ),
    torch.tensor(
        y_full_scaled,
        dtype=torch.float32
    )
)

final_generator = torch.Generator()
final_generator.manual_seed(SEED)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=FINAL_DL_CONFIG[
        "batch_size"
    ],
    shuffle=True,
    generator=final_generator
)

# initialize selected model
final_dl_model = TunedWiderMLP(
    input_size=X_full_processed.shape[1],
    output_size=len(TARGETS),
    activation=FINAL_DL_CONFIG[
        "activation"
    ]
).to(device)

loss_function = nn.MSELoss()

optimizer = torch.optim.AdamW(
    final_dl_model.parameters(),
    lr=FINAL_DL_CONFIG[
        "learning_rate"
    ],
    weight_decay=FINAL_DL_CONFIG[
        "weight_decay"
    ]
)

# adapt learning rate using training loss
scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=1e-5
    )
)

training_history = []
training_start = time.time()

# train on the complete training set
for epoch in range(
    1,
    FINAL_DL_CONFIG["epochs"] + 1
):

    final_dl_model.train()
    total_train_loss = 0.0

    for (
        predictor_batch,
        target_batch
    ) in final_train_loader:

        predictor_batch = (
            predictor_batch.to(device)
        )

        target_batch = (
            target_batch.to(device)
        )

        optimizer.zero_grad()

        batch_predictions = final_dl_model(
            predictor_batch
        )

        loss = loss_function(
            batch_predictions,
            target_batch
        )

        loss.backward()
        optimizer.step()

        total_train_loss += (
            loss.item()
            * len(target_batch)
        )

    train_loss = (
        total_train_loss
        / len(final_train_dataset)
    )

    scheduler.step(train_loss)

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "learning_rate": current_lr
    })

    if (
        epoch == 1
        or epoch % 10 == 0
        or epoch == FINAL_DL_CONFIG["epochs"]
    ):

        elapsed = (
            time.time()
            - training_start
        ) / 60

        print(
            f"Epoch {epoch:3d}/"
            f"{FINAL_DL_CONFIG['epochs']} | "
            f"train loss={train_loss:.4f} | "
            f"lr={current_lr:.6f} | "
            f"time={elapsed:.1f} minutes"
        )

# predict test data
test_predictions_scaled = (
    predict_tuned_mlp(
        final_dl_model,
        X_test_processed
    )
)

test_predictions_transformed = (
    test_predictions_scaled
    * final_target_std
    + final_target_mean
)

dl_test_predictions = (
    inverse_signed_log1p(
        test_predictions_transformed
    )
)

# apply accounting relationships
dl_test_predictions = (
    apply_tuning_accounting(
        dl_test_predictions,
        X_test
    )
)

# create prediction file
final_dl_predictions = pd.DataFrame(
    dl_test_predictions,
    columns=TARGETS
)

final_dl_predictions.insert(
    0,
    ID_COLUMN,
    test_ids.to_numpy()
)

final_dl_predictions.to_csv(
    DL_SAVE_DIRECTORY
    / "final_dl_predictions.csv",
    index=False
)

# save preprocessing pipeline
joblib.dump(
    final_dl_preprocessor,
    DL_SAVE_DIRECTORY
    / "dl_preprocessor.joblib",
    compress=3
)

# save model and settings
torch.save(
    {
        "state_dict": (
            final_dl_model
            .state_dict()
        ),
        "input_size": (
            X_full_processed.shape[1]
        ),
        "output_size": len(TARGETS),
        "target_mean": final_target_mean,
        "target_std": final_target_std,
        "targets": TARGETS,
        "id_column": ID_COLUMN,
        "config": FINAL_DL_CONFIG,
        "seed": SEED,
        "training_history": (
            training_history
        )
    },
    DL_SAVE_DIRECTORY
    / "final_dl_model.pt"
)

# save predictions separately
np.save(
    DL_SAVE_DIRECTORY
    / "final_dl_test_predictions.npy",
    dl_test_predictions
)

print(
    "\nTraining completed in "
    f"{(time.time() - training_start) / 60:.1f} minutes"
)

print("\nSaved files:")

for file_path in (
    DL_SAVE_DIRECTORY.iterdir()
):
    print(
        f"{file_path.name}: "
        f"{file_path.stat().st_size / 1_000_000:.1f} MB"
    )

print(
    "\nPrediction shape:",
    final_dl_predictions.shape
)

display(
    final_dl_predictions.head()
)

Selected DL configuration:
{'name': 'Wider MLP - Leaky ReLU', 'hidden_layers': [1024, 512, 256, 128], 'activation': 'leaky_relu', 'dropout': 0.1, 'weight_decay': 0.0001, 'learning_rate': 0.001, 'epochs': 200, 'batch_size': 1024}
Device: cpu

Preprocessing predictors...
Processed training shape: (100000, 332)
Processed test shape: (50000, 332)
Epoch   1/200 | train loss=0.4441 | lr=0.001000 | time=0.1 minutes
Epoch  10/200 | train loss=0.1628 | lr=0.001000 | time=1.3 minutes
Epoch  20/200 | train loss=0.1418 | lr=0.001000 | time=2.7 minutes
Epoch  30/200 | train loss=0.1320 | lr=0.001000 | time=4.2 minutes
Epoch  40/200 | train loss=0.1250 | lr=0.001000 | time=5.6 minutes
Epoch  50/200 | train loss=0.1210 | lr=0.001000 | time=7.1 minutes
Epoch  60/200 | train loss=0.1175 | lr=0.001000 | time=8.5 minutes
Epoch  70/200 | train loss=0.1129 | lr=0.001000 | time=10.0 minutes
Epoch  80/200 | train loss=0.1091 | lr=0.001000 | time=11.5 minutes
Epoch  90/200 | train loss=0.1071 | lr=0.001000 | 

,Id,Q0_TOTAL_ASSETS,Q0_TOTAL_LIABILITIES,Q0_TOTAL_STOCKHOLDERS_EQUITY,Q0_GROSS_PROFIT,Q0_COST_OF_REVENUES,Q0_REVENUES,Q0_OPERATING_INCOME,Q0_OPERATING_EXPENSES,Q0_EBITDA
0,171381,"163,669,019.8510","53,428,032.0875","118,481,926.4685","7,441,878.7148","10,098,054.0459","17,539,932.7607","-8,615,859.8561","16,057,738.5709","-8,779,234.8919"
1,171809,"67,573,249.7042","25,864,556.9480","40,673,511.7627","4,486,668.0731","6,581,326.8613","11,067,994.9344","-4,832,735.7945","9,319,403.8677","-4,777,366.2824"
2,109665,"24,623,474.0310","16,126,829.7541","2,019,163.1540","-233,432.5074","1,497,508.0038","1,264,075.4965","-2,746,528.0036","2,513,095.4962","-2,757,649.5064"
3,122380,"90,494,775.4843","37,253,700.6623","39,762,014.2043","1,148,520.3275","1,977,697.1044","3,126,217.4320","-7,737,402.7569","8,885,923.0844","-6,695,642.2051"
4,140286,"237,535,044.2190","88,544,769.1851","145,657,170.9474","5,706,859.0426","12,124,415.7170","17,831,274.7596","-20,847,427.1015","26,554,286.1440","-20,790,937.9546"


# Final Conclusion

This project developed and compared three major types of forecasting approaches using the same five-fold cross-validation procedure: a naive mean baseline, several non-deep-learning models, and multiple deep-learning architectures. All models were evaluated using average target-level sMAPE in the original target units. A signed-log transformation was applied during training to reduce the effect of extreme values while preserving negative financial values.

## Model Comparison

| Model | Mean CV sMAPE | CV Standard Deviation |
|---|---:|---:|
| Mean baseline | 135.9331 | 0.4741 |
| Elastic Net | 94.6253 | 0.2182 |
| Random Forest | 27.3906 | 0.1270 |
| XGBoost | 23.2273 | 0.1455 |
| Accounting-adjusted XGBoost | 14.0320 | 0.1024 |
| XGBoost–Random Forest ensemble | **13.9664** | **0.1020** |
| Wider MLP with ReLU | 26.5376 | 0.3040 |
| Hybrid GRU–MLP | 28.7174 | 0.5319 |
| Wider MLP with Leaky ReLU and weight decay | 25.7305 | 0.2183 |

Elastic Net performed poorly, indicating that the relationships between the predictors and financial targets were too nonlinear for a regularized linear model. Random Forest provided a large improvement, while XGBoost produced the strongest single-model result.

The largest improvement came from applying accounting relationships after prediction. Gross profit was calculated as predicted revenue minus predicted cost of revenues, operating income was calculated as gross profit minus predicted operating expenses, and EBITDA was calculated using operating income and Q1 depreciation and amortization. This reduced the XGBoost sMAPE from **23.2273 to 14.0320**. Predicting Q0 depreciation and amortization with Ridge and XGBoost did not improve performance, as simply carrying forward Q1 D&A was more accurate.

A weighted ensemble containing **93% accounting-adjusted XGBoost and 7% accounting-adjusted Random Forest** achieved the best overall result of **13.9664 ± 0.1020**. The small Random Forest contribution provided complementary predictions, but XGBoost remained the dominant component.

## Deep-Learning Findings

The deep-learning experiments showed that network width was more useful than additional depth. The Wider Deep MLP outperformed the reference and deeper architectures, while the Hybrid GRU–MLP did not provide an advantage. The GRU may have compressed useful quarterly information into a representation that was less informative than directly providing all quarterly predictors to an MLP.

Dropout, learning-rate scheduling, checkpoint selection, activation functions, and weight decay were also evaluated. The ReLU model with weight decay achieved the lowest mean sMAPE, but the **Leaky ReLU model with 10% dropout and weight decay of $10^{-4}$** provided a better balance between accuracy and cross-validation stability. It was therefore selected as the final deep-learning configuration.

The selected DL model was fitted on the complete training dataset for 200 epochs. Its fitted preprocessing pipeline, model weights, training history, and test predictions were saved so that the model can be reused without retraining.

## Overall Findings

The main conclusions are:

- Nonlinear models substantially outperformed the naive baseline and Elastic Net
- XGBoost was the strongest individual model for this structured financial dataset
- Accounting-based post-processing produced a larger improvement than further model complexity
- A small Random Forest contribution improved the stability and accuracy of XGBoost
- Wider neural networks performed better than deeper or recurrent architectures
- Weight decay and Leaky ReLU improved the stability of the selected deep-learning model
- Operating income and EBITDA remained the most difficult targets because they frequently contain negative or near-zero values and accumulate errors from other predicted financial quantities

Based on cross-validation performance, the **93% XGBoost–7% Random Forest ensemble** was selected as the final overall model. The test dataset was used only after model selection and full-data training. Because the test targets are hidden, no test sMAPE was calculated, preventing test-data leakage.